# Indicadores de seguimiento de la evaluación del PNIE <br> Parte III: Necesidades de intervención

Autor: Equipo de Evaluación del PNIE

Fecha de creación: 29/08/2025

Fecha de actualización: 07/10/2025

## 01. Configuración del notebook

In [4]:
# Instalando librerias
# !pip install dbfread

In [5]:
# Importando librerias
import os
import numpy as np
import pandas as pd
from dbfread import DBF

In [6]:
# Directorio de trabajo
file1 = 'E:/OneDrive - Ministerio de Educación/00. Paolo/Actividades'
file1 = 'D:/OneDrive - Ministerio de Educación/00. Paolo/Actividades'
file2 = '01. Evaluación PNIE/02. IS-PNIE'
file = file1 + '/' + file2
os.chdir(file)

In [7]:
# Configurando el formato
pd.options.mode.chained_assignment = None
pd.options.display.float_format = '{:,.1f}'.format

In [8]:
# Definiendo funciones de conteo
def unique(data, varlist):
    aux = data[varlist]
    a = aux.drop_duplicates().shape[0]
    list = " ".join(varlist)
    print(f'Number of unique values of {list} is {a:,.0f}')
    print(f'Number of records is {data.shape[0]:,.0f}')

def by_unique(data, varlist, var):
    unique(data, varlist)
    list = varlist + var
    aux = data[list]
    a = aux.drop_duplicates().value_counts(var).reset_index().\
    sort_values(by=var)
    print("\n", a)

In [9]:
# Definiendo funciones de necesidad de intervencion
def iden_nec_prod(data, inds, new_ind):
    data[new_ind] = data[inds].gt(0).any(axis=1).astype('int8')

In [10]:
# Corrección de codigos locales
def codlocal(df, oldvar, newvar):
    df[newvar] = df[oldvar].astype(str).str.zfill(6)

In [11]:
def freq(df, columna, dropna=False, decimales=1):
    """
    Genera una tabla de frecuencias absolutas, relativas (%) y totales.

    Parámetros:
    -----------
    df : DataFrame
        El DataFrame de entrada
    columna : str
        Nombre de la columna a analizar
    dropna : bool, opcional
        Si True, ignora los valores NaN. Default = False
    decimales : int, opcional
        Número de decimales para el porcentaje. Default = 2
    """
    
    # Frecuencias absolutas y relativas
    freq = df[columna].value_counts(dropna=dropna)
    rel = df[columna].value_counts(normalize=True, dropna=dropna) * 100
    
    # Construcción de la tabla
    tabla = pd.DataFrame({
        'Freq': freq,
        '(%)': rel.round(decimales)
    })
    
    # Totales
    total_freq = tabla['Freq'].sum()
    tabla.loc['TOTAL'] = [total_freq, 100]

    # Formateo: Freq con separador de miles y (%) con % explícito
    tabla['Freq'] = tabla['Freq'].apply(lambda x: f"{int(x):,}" if pd.notna(x) else x)
    tabla['(%)'] = tabla['(%)'].apply(lambda x: f"{x:.{decimales}f}%" if x != 100 else "100%")
 
    return print(tabla, '\n')

In [12]:
def ind_nec(df, out_col, cols):
    pair = df[list(cols)]
    df[out_col] = np.where(
        pair.gt(0).any(axis=1), 1,
        np.where(pair.isna().all(axis=1), np.nan, 0)
    )

## 02. Año 2017 - PNIE

### 2.1 Productos

#### 2.1.1 Indicadores de subproductos

##### A. Necesidad de intervenciones en infraestructura

In [88]:
file = 'data/LE_BasePr_PNIE.dta'
df_0 = pd.read_stata(file)
df = df_0.copy()

# Corrigiendo codigos locales
codlocal(df, 'id_local', 'codlocal')

In [89]:
# GI1
ind_nec(df, 'GI1_1', ['CTot_locdem', 'CTot_edifdem'])
ind_nec(df, 'GI1_2', ['CTot_refinc', 'CTot_refconv'])
ind_nec(df, 'GI1_3', ['CTot_icont_p3g1'])

clima = ['Tropical húmedo', 'Subtropical húmedo', 'Ceja de montaña']
df['GI1_4'] = pd.NA
elig = (df['area_sig'] == 'Rural') & (df['clima'].isin(clima))
df.loc[elig, 'GI1_4'] = 0
df.loc[elig & df['CTot_PS'].gt(0), 'GI1_4'] = 1
df['GI1_4'] = df['GI1_4'].astype('Int8')

ind_nec(df, 'GI1_5', ['Ctot_cerco'])

# GI2
ind_nec(df, 'GI2_2', ['ctot_calidadays_aj'])

# GI3
ind_nec(df, 'GI3_1', ['CT_CalidadEle'])
ind_nec(df, 'GI3_2', ['Mobiliario_E'])
ind_nec(df, 'GI3_3', ['CTot_ENE'])

df['cu'] = df['costo_prom'].min()
df['fcr'] = np.where(df['area_sig'] == 'Urbana', 1, 0.94)
df['area'] = df['areatechada'] - (df['area_dem'] + df['area_ri'] + \
                                  df['area_rc']) + 0.0001
df.loc[df['area'] < 0, 'area'] = 0
df['val_act'] = df['area'] * df['cu'] * df['fcr'] * 1.05 * 1.20 * 1.18
df['GI3_4'] = df['val_act'] * 0.01
mask = ((df['COSTO_TOTAL'].isna()) | (df['CTot_repLOC'].gt(0)))
df.loc[mask, 'GI3_4'] = np.nan
df.drop(columns=['cu', 'fcr', 'area', 'val_act'], inplace=True)

# GI4
ind_nec(df, 'GI4_1', ['CTot_repo_edifdem'])
ind_nec(df, 'GI4_3', ['Ctot_accesibilidad'])
ind_nec(df, 'GI4_4', ['CTot_ampLOC'])

# GI5
ind_nec(df, 'GI5_1', ['CTot_repLOC'])

##### B. Acceso adecuado a servicios básicos

###### Energía eléctrica

In [90]:
file = 'data/CIE/P2_D_1N.dta'
sb_0 = pd.read_stata(file)

In [91]:
sb = sb_0.copy()

# Corrigiendo codigos locales
codlocal(sb, 'id_local', 'codlocal')

# Identificando procedencia de energía eléctrica
sb['p2_d_1_cod_est'] = sb['p2_d_1_cod_est'].astype('string')
sb.loc[sb['p2_d_1_cod_est'] == 'si', 'p2_d_1_cod_est'] = '1'
sb.loc[sb['p2_d_1_cod_est'] == 'no', 'p2_d_1_cod_est'] = '0'
sb.loc[sb['p2_d_1_cod_est'].isna(), 'p2_d_1_cod_est'] = '9'
sb['p2_d_1_cod_est'] = sb['p2_d_1_cod_est'].astype('int')

# Colapsando a nivel de codlocal-predio
sb = sb.pivot_table(index=['codlocal', 'nro_pred'],
                    columns='p2_d_1_cod',
                    values='p2_d_1_cod_est',
                    aggfunc='max', observed=True).reset_index()
sb.columns.name = None

sb['min'] = sb[['Red Publica', 'Generador o Motor', 'panel solar', 
                'otros']].min(axis=1)
sb['max'] = sb[['Red Publica', 'Generador o Motor', 'panel solar', 
                'otros']].max(axis=1)
sb['sum'] = sb[['Red Publica', 'Generador o Motor', 'panel solar', 
                'otros']].sum(axis=1)

print(pd.crosstab(sb['min'], sb['max']))
print(sb['sum'].value_counts(dropna=False))

# Colapsando a nivel de codlocal
sb = sb.drop(columns=['min', 'max', 'sum'], errors='ignore')
sb = sb.groupby('codlocal')[['Red Publica', 'Generador o Motor', 'panel solar', 
                             'otros']].max().reset_index()

# Identificando el tipo de abastecimiento
sb['sum'] = sb[['Red Publica', 'Generador o Motor', 'panel solar', 'otros']]. \
    sum(axis=1)
print(sb['sum'].value_counts(dropna=False))

sb['tipo_ee'] = ""
sb.loc[sb['sum'] == 36, 'tipo_ee'] = '5. Sin información'
sb.loc[(sb['sum'] == 1) & (sb['Red Publica'] == 1), 
       'tipo_ee'] = '1. Red pública'
sb.loc[(sb['sum'] == 1) & (sb['Generador o Motor'] == 1), 
       'tipo_ee'] = '2. Generador'
sb.loc[(sb['sum'] == 1) & (sb['panel solar'] == 1), 
       'tipo_ee'] = '3. Panel solar'
sb.loc[(sb['sum'] == 1) & (sb['otros'] == 1), 
       'tipo_ee'] = '4. Otros'
sb.loc[(sb['sum'] == 2) & (sb['Red Publica'] == 1), 
       'tipo_ee'] = '1. Red pública'
sb.loc[(sb['sum'] == 2) & (sb['panel solar'] == 1), 
       'tipo_ee'] = '3. Panel solar'
print(sb['tipo_ee'].value_counts(dropna=False))

# Identificando acceso adecuado
adec = ['1. Red pública',
        '3. Panel solar',
        '4. Energía eólica']
sb['GI4_2'] = sb['tipo_ee'].isin(adec).astype('Int8')
sb.loc[sb['tipo_ee'] == '5. Sin información', 'GI4_2'] = pd.NA
elec = sb.copy()

max      1      9
min              
0    31891      0
9        0  11192
sum
1     31878
36    11192
2        13
Name: count, dtype: int64
sum
1     30021
36    10814
2        13
Name: count, dtype: int64
tipo_ee
1. Red pública        29846
5. Sin información    10814
2. Generador            106
4. Otros                 44
3. Panel solar           38
Name: count, dtype: int64


###### Agua

In [92]:
file = 'data/CIE/P2_D_5N.dta'
sb_0 = pd.read_stata(file)

In [93]:
sb = sb_0.copy()

# Corrigiendo codigos locales
codlocal(sb, 'id_local', 'codlocal')

# Identificando procedencia de agua
sb['p2_d_5_cod_est'] = sb['p2_d_5_cod_est'].astype('string')
sb.loc[sb['p2_d_5_cod_est'] == 'si', 'p2_d_5_cod_est'] = '1'
sb.loc[sb['p2_d_5_cod_est'] == 'no', 'p2_d_5_cod_est'] = '0'
sb.loc[sb['p2_d_5_cod_est'].isna(), 'p2_d_5_cod_est'] = '9'
sb['p2_d_5_cod_est'] = sb['p2_d_5_cod_est'].astype('int')

# Colapsando a nivel de codlocal-predio
sb = sb.pivot_table(index=['codlocal', 'nro_pred'],
                    columns='p2_d_5_cod',
                    values='p2_d_5_cod_est',
                    aggfunc='max', observed=True).reset_index()
sb.columns.name = None

sb['min'] = sb[['Red publica',
                'Pilon de uso publico (agua potable)', 
                'Camion cisterna u otro similar',
                'Pozo', 'Rio, acequia, manantial o similar', 
                'Otro']].min(axis=1)
sb['max'] = sb[['Red publica',
                'Pilon de uso publico (agua potable)', 
                'Camion cisterna u otro similar',
                'Pozo', 'Rio, acequia, manantial o similar', 
                'Otro']].max(axis=1)
sb['sum'] = sb[['Red publica',
                'Pilon de uso publico (agua potable)', 
                'Camion cisterna u otro similar',
                'Pozo', 'Rio, acequia, manantial o similar', 
                'Otro']].sum(axis=1)

print(pd.crosstab(sb['min'], sb['max']))
print(sb['sum'].value_counts(dropna=False))

# Colapsando a nivel de codlocal
sb = sb.drop(columns=['min', 'max', 'sum'], errors='ignore')
sb = sb.groupby('codlocal')[['Red publica',
                             'Pilon de uso publico (agua potable)',
                             'Camion cisterna u otro similar',
                             'Pozo', 'Rio, acequia, manantial o similar',
                             'Otro']].max().reset_index()

# Identificando el tipo de abastecimiento
sb['sum'] = sb[['Red publica',
                'Pilon de uso publico (agua potable)', 
                'Camion cisterna u otro similar',
                'Pozo', 'Rio, acequia, manantial o similar', 
                'Otro']].sum(axis=1)
print(sb['sum'].value_counts(dropna=False))

sb['tipo_agua'] = ""
sb.loc[sb['sum'] == 54, 'tipo_agua'] = '7. Sin información'
sb.loc[(sb['sum'] == 1) & (sb['Red publica'] == 1), 
       'tipo_agua'] = '1. Red pública'
sb.loc[(sb['sum'] == 1) & (sb['Pilon de uso publico (agua potable)'] == 1), 
       'tipo_agua'] = '2. Pilón de uso público'
sb.loc[(sb['sum'] == 1) & (sb['Camion cisterna u otro similar'] == 1), 
       'tipo_agua'] = '3. Camión cisterna u otro similar'
sb.loc[(sb['sum'] == 1) & (sb['Pozo'] == 1), 
       'tipo_agua'] = '4. Pozo'
sb.loc[(sb['sum'] == 1) & (sb['Rio, acequia, manantial o similar'] == 1), 
       'tipo_agua'] = '5. Río, acequia, manantial o similar'
sb.loc[(sb['sum'] == 1) & (sb['Otro'] == 1), 
       'tipo_agua'] = '6. Otros'
sb.loc[(sb['sum'] == 2) & (sb['Red publica'] == 1), 
       'tipo_agua'] = '1. Red pública'
sb.loc[(sb['sum'] == 2) & (sb['Pilon de uso publico (agua potable)'] == 1), 
       'tipo_agua'] = '2. Pilón de uso público'
sb.loc[(sb['sum'] == 2) & (sb['Camion cisterna u otro similar'] == 1), 
       'tipo_agua'] = '3. Camión cisterna u otro similar'
sb.loc[(sb['sum'] == 2) & (sb['Pozo'] == 1), 
       'tipo_agua'] = '4. Pozo'
sb.loc[(sb['sum'] == 2) & (sb['Rio, acequia, manantial o similar'] == 1), 
       'tipo_agua'] = '5. Río, acequia, manantial o similar'
sb.loc[(sb['sum'] == 3) & (sb['Red publica'] == 1), 
       'tipo_agua'] = '1. Red pública'
sb.loc[(sb['sum'] == 3) & (sb['Pilon de uso publico (agua potable)'] == 1), 
       'tipo_agua'] = '2. Pilón de uso público'
sb.loc[(sb['sum'] == 3) & (sb['Camion cisterna u otro similar'] == 1), 
       'tipo_agua'] = '3. Camión cisterna u otro similar'
sb.loc[(sb['sum'] == 3) & (sb['Pozo'] == 1), 
       'tipo_agua'] = '4. Pozo'
print(sb['tipo_agua'].value_counts(dropna=False))

# Identificando acceso adecuado
adec = ['1. Red pública',
        '2. Pilón de uso público',
        '3. Camión cisterna u otro similar']
sb['agua_adecuado'] = sb['tipo_agua'].isin(adec).astype('Int8')
agua = sb.copy()

max      1    9
min            
0    42838    0
9        0  245
sum
1     41988
2       839
54      245
3        11
Name: count, dtype: int64
sum
1     39383
2      1193
54      244
3        28
Name: count, dtype: int64
tipo_agua
5. Río, acequia, manantial o similar    16694
1. Red pública                          15246
4. Pozo                                  3849
6. Otros                                 3070
2. Pilón de uso público                  1319
3. Camión cisterna u otro similar         426
7. Sin información                        244
Name: count, dtype: int64


###### Saneamiento

In [94]:
file = 'data/CIE/P2_D_9N.dta'
sb_0 = pd.read_stata(file)

In [95]:
sb = sb_0.copy()

# Corrigiendo codigos locales
codlocal(sb, 'id_local', 'codlocal')

# Identificando procedencia de agua
sb['p2_d_9_cod'] = sb['p2_d_9_cod'].astype('string')
sb.loc[sb['p2_d_9_cod'] == 'si', 'p2_d_9_cod'] = '1'
sb.loc[sb['p2_d_9_cod'] == 'no', 'p2_d_9_cod'] = '0'
sb.loc[sb['p2_d_9_cod'].isna(), 'p2_d_9_cod'] = '9'
sb['p2_d_9_cod'] = sb['p2_d_9_cod'].astype('int')

# Colapsando a nivel de codlocal-predio
sb = sb.pivot_table(index=['codlocal', 'nro_pred'],
                    columns='p2_d_9_nro',
                    values='p2_d_9_cod',
                    aggfunc='max', observed=True).reset_index()
sb.columns.name = None

sb['min'] = sb[['Red publica', 'Tanque septico y pozo percolador', 
                'Pozo con tratamiento', 'Pozo sin tratamiento', 'Rio acequia', 
                'Zanja filtrante', 'No tiene']].min(axis=1)
sb['max'] = sb[['Red publica', 'Tanque septico y pozo percolador', 
                'Pozo con tratamiento', 'Pozo sin tratamiento', 'Rio acequia', 
                'Zanja filtrante', 'No tiene']].max(axis=1)
sb['sum'] = sb[['Red publica', 'Tanque septico y pozo percolador', 
                'Pozo con tratamiento', 'Pozo sin tratamiento', 'Rio acequia', 
                'Zanja filtrante', 'No tiene']].sum(axis=1)

print(pd.crosstab(sb['min'], sb['max']))
print(sb['sum'].value_counts(dropna=False))

# Colapsando a nivel de codlocal
sb = sb.drop(columns=['min', 'max', 'sum'], errors='ignore')
sb = sb.groupby('codlocal')[['Red publica', 'Tanque septico y pozo percolador', 
                             'Pozo con tratamiento', 'Pozo sin tratamiento', 
                             'Rio acequia', 'Zanja filtrante', 
                             'No tiene']].max().reset_index()

# Identificando el tipo de abastecimiento
sb['sum'] = sb[['Red publica', 'Tanque septico y pozo percolador', 
                'Pozo con tratamiento', 'Pozo sin tratamiento', 'Rio acequia', 
                'Zanja filtrante', 'No tiene']].sum(axis=1)
print(sb['sum'].value_counts(dropna=False))

sb['tipo_sane'] = ""
sb.loc[sb['sum'] == 63, 'tipo_sane'] = '8. Sin información'
sb.loc[(sb['sum'] == 1) & (sb['Red publica'] == 1), 
       'tipo_sane'] = '1. Red pública'
sb.loc[(sb['sum'] == 1) & (sb['Tanque septico y pozo percolador'] == 1), 
       'tipo_sane'] = '2. Tanque séptico y pozo percolador'
sb.loc[(sb['sum'] == 1) & (sb['Pozo con tratamiento'] == 1), 
       'tipo_sane'] = '3. Pozo con tratamiento'
sb.loc[(sb['sum'] == 1) & (sb['Pozo sin tratamiento'] == 1), 
       'tipo_sane'] = '4. Pozo sin tratamiento'
sb.loc[(sb['sum'] == 1) & (sb['Rio acequia'] == 1), 
       'tipo_sane'] = '5. Río acequia'
sb.loc[(sb['sum'] == 1) & (sb['Zanja filtrante'] == 1), 
       'tipo_sane'] = '6. Zanja filtrante'
sb.loc[(sb['sum'] == 1) & (sb['No tiene'] == 1), 
       'tipo_sane'] = '7. No tiene'
sb.loc[(sb['sum'] == 2) & (sb['Red publica'] == 1), 
       'tipo_sane'] = '1. Red pública'
sb.loc[(sb['sum'] == 2) & (sb['Tanque septico y pozo percolador'] == 1), 
       'tipo_sane'] = '2. Tanque séptico y pozo percolador'
sb.loc[(sb['sum'] == 2) & (sb['Pozo con tratamiento'] == 1), 
       'tipo_sane'] = '3. Pozo con tratamiento'
sb.loc[(sb['sum'] == 2) & (sb['Pozo sin tratamiento'] == 1), 
       'tipo_sane'] = '4. Pozo sin tratamiento'
sb.loc[(sb['sum'] == 2) & (sb['Rio acequia'] == 1), 
       'tipo_sane'] = '5. Río acequia'
sb.loc[(sb['sum'] == 2) & (sb['Zanja filtrante'] == 1), 
       'tipo_sane'] = '6. Zanja filtrante'
sb.loc[(sb['sum'] == 3) & (sb['Red publica'] == 1), 
       'tipo_sane'] = '1. Red pública'
sb.loc[(sb['sum'] == 3) & (sb['Tanque septico y pozo percolador'] == 1), 
       'tipo_sane'] = '2. Tanque séptico y pozo percolador'
sb.loc[(sb['sum'] == 3) & (sb['Pozo con tratamiento'] == 1), 
       'tipo_sane'] = '3. Pozo con tratamiento'
sb.loc[(sb['sum'] == 3) & (sb['Pozo sin tratamiento'] == 1), 
       'tipo_sane'] = '4. Pozo sin tratamiento'
sb.loc[(sb['sum'] == 3) & (sb['Rio acequia'] == 1), 
       'tipo_sane'] = '5. Río acequia'
sb.loc[(sb['sum'] == 4) & (sb['Red publica'] == 1), 
       'tipo_sane'] = '1. Red pública'
print(sb['tipo_sane'].value_counts(dropna=False))

# Identificando acceso adecuado
adec = ['1. Red pública',
        '2. Tanque séptico y pozo percolador']
sb['sane_adecuado'] = sb['tipo_sane'].isin(adec).astype('Int8')
sane = sb.copy()

max      1    9
min            
0    42646    0
9        0  437
sum
1     42198
2       441
63      437
3         7
Name: count, dtype: int64
sum
1     39250
2      1136
63      433
3        28
4         1
Name: count, dtype: int64
tipo_sane
1. Red pública                         12679
7. No tiene                             8724
4. Pozo sin tratamiento                 7919
2. Tanque séptico y pozo percolador     6413
3. Pozo con tratamiento                 2632
5. Río acequia                          1765
8. Sin información                       433
6. Zanja filtrante                       283
Name: count, dtype: int64


###### Consolidación de SSBB

In [96]:
sb = pd.merge(elec[['codlocal', 'GI4_2']], 
              agua[['codlocal', 'tipo_agua', 'agua_adecuado']],
              on='codlocal', how='outer')
sb = pd.merge(sb, sane[['codlocal', 'tipo_sane', 'sane_adecuado']],
              on='codlocal', how='outer')

sb['GI2_1'] = ((sb['agua_adecuado'] == 1) & 
               (sb['sane_adecuado'] == 1)).astype('Int8')

# Eliminando observaciones sin información de servicios básicos
sb.loc[sb['tipo_agua'] == '7. Sin información', 'GI2_1'] = pd.NA
sb.loc[sb['tipo_sane'] == '8. Sin información', 'GI2_1'] = pd.NA

# Seleccionando variables
sb_vf = sb[['codlocal', 'GI2_1', 'GI4_2']]
sb_vf.sort_values('codlocal', inplace=True)
sb_vf = sb_vf.reset_index(drop=True)

##### C. Consolidación

In [97]:
# Uniendo bases
df2 = pd.merge(df, sb_vf, on='codlocal', how='left')

# Seleccionando variables
df = df2[['codlocal', 'COSTO_TOTAL',
             'GI1_1', 'GI1_2', 'GI1_3', 'GI1_4', 'GI1_5',
             'GI2_1', 'GI2_2',
             'GI3_1', 'GI3_2', 'GI3_3', 'GI3_4',
             'GI4_1', 'GI4_2', 'GI4_3', 'GI4_4',
             'GI5_1']]

#### 2.1.2 Indicadores de productos

In [98]:
df = df.copy()

In [99]:
# GI1
iden_nec_prod(df, ['GI1_1', 'GI1_2', 'GI1_3', 'GI1_4', 'GI1_5'], 'GI1')

# GI2
iden_nec_prod(df, ['GI2_1', 'GI2_2'], 'GI2')
df.loc[((df['GI2_1'] == 0) & (df['GI2_2'] == 0)), 'GI2'] = 1

# GI3
iden_nec_prod(df, ['GI3_1', 'GI3_2', 'GI3_3', 'GI3_4'], 'GI3')

# GI4
iden_nec_prod(df, ['GI4_1', 'GI4_2', 'GI4_3', 'GI4_4'], 'GI4')
df.loc[((df['GI4_2'] == 0) & (df['GI4'] == 0)), 'GI4'] = 1

### 2.2 Actividades

In [100]:
# Universo de LLLEE del año
file = 'data/LE_BasePr_PNIE.dta'
df_0 = pd.read_stata(file)

# Lista de LLEE "existentes"
file = 'data/procesadas/llee_existentes.csv'
exis_0 = pd.read_csv(file, dtype={'codlocal': str})

In [101]:
sfl = df_0.copy()
exis = exis_0.copy()

# Corrigiendo codigos locales
codlocal(sfl, 'id_local', 'codlocal')

# Identificando LLEE "existentes" y "nuevos"
sfl = pd.merge(sfl, exis,
               how='left',
               on='codlocal',
               indicator='union')
sfl['union'].value_counts(dropna=False)

# Supuesto: Todos los LLEE del PNIE son considerados existentes
sfl.loc[sfl['union'] == 'left_only', 'union'] = 'both'

# Estado del SFL de los predios
freq(sfl, 'estado_SAFIL')

                                       Freq    (%)
estado_SAFIL                                      
No saneado (Saneable largo plazo)    15,656  37.0%
Saneado                              12,947  30.6%
No saneado (Saneable mediano plazo)   7,486  17.7%
No saneado (Saneable corto plazo)     4,144   9.8%
                                      1,250   3.0%
sin clasificacion                       848   2.0%
TOTAL                                42,331   100% 



#### 2.2.1 ET.1 SFL de predios existentes

In [102]:
sfl.loc[sfl['union'] == 'both', 'ET1'] = 0
sfl.loc[(sfl['ET1'] == 0) & (sfl['estado_SAFIL'] == 'Saneado'), 'ET1'] = 1
sfl.loc[sfl['estado_SAFIL'] == '', 'ET1'] = np.nan

freq(sfl, 'ET1')

          Freq    (%)
ET1                  
0.0     28,134  66.5%
1.0     12,947  30.6%
 NaN     1,250   3.0%
 TOTAL  42,331   100% 



#### 2.2.2 ET.2 SFL de predios nuevos

In [103]:
sfl.loc[sfl['union'] == 'left_only', 'ET2'] = 0
sfl.loc[(sfl['ET2'] == 0) & (sfl['estado_SAFIL'] == 'Saneado'), 'ET2'] = 1
sfl.loc[sfl['estado_SAFIL'] == '', 'ET2'] = np.nan

freq(sfl, 'ET2')

         Freq   (%)
ET2                
NaN    42,331  100%
TOTAL  42,331  100% 



#### 2.2.3 Consolidación

In [104]:
# Seleccionando variables
sfl = sfl[['codlocal', 'ET1', 'ET2']]

# Consolidando información
df = pd.merge(df, sfl,
              on='codlocal',
              how='left')

### 2.3 Resultados específicos

In [105]:
iden_nec_prod(df, ['GI1', 'GI2', 'GI3_1', 'GI3_2', 'GI4', 'GI5_1', 'ET1'],
              'OE1')
iden_nec_prod(df, ['GI3_3', 'GI3_4'],
              'OE4')

### 2.4 Resultados finales

In [106]:
file = 'data/SIAGIE Reporte Matricula 2013.dbf'
table = DBF(file, encoding='cp437', lowernames=True, 
            ignore_missing_memofile=True)
matri_0 = pd.DataFrame(iter(table))

In [107]:
matri = matri_0.copy()

# Estimando la matricula en 2013 (solo se tiene información EBR)
codlocal(matri, 'codigoloca', 'codlocal')
matri = matri[matri['gestion'].ne('3')]
matri = matri.groupby('codlocal')['total'].sum().reset_index()
matri.rename(columns={'total': 'matricula'}, inplace=True)

In [108]:
df = pd.merge(df, matri, how='left', on='codlocal')

df['RF1'] = 0

# Supuesto: En la línea base del PNIE no se identificaron LL.EE. con brecha
# cerrada.

df['RF2_matri'] = 0
df.loc[df['RF1'] == 1, 'RF2_matri'] = df['matricula']

In [109]:
# Añadiendo año del diagnóstico
sufix = 'PNIE'
df.rename(columns={c: f'{c}_{sufix}' for c in df.columns[1:]}, 
          inplace=True)

# Exportando a csv
df.sort_values('codlocal', inplace=True)
df = df.reset_index(drop=True)
df.to_csv('data/procesadas/necint_PNIE.csv', index=False)

## 03. Año 2019

### 3.1 Productos

#### 3.1.1 Indicadores de subproductos

##### A. Necesidad de intervenciones en infraestructura

In [110]:
file = 'data/LE_BasePr_2019.dta'
df_0 = pd.read_stata(file)
df = df_0.copy()

# Corrigiendo codigos locales
codlocal(df, 'cod_local', 'codlocal')

In [111]:
# GI1
ind_nec(df, 'GI1_1', ['ct_st_d', 'ct_sp_d'])
ind_nec(df, 'GI1_2', ['ct_ri', 'ct_rc'])
ind_nec(df, 'GI1_3', ['ct_ic'])

clima = ['Tropical húmedo', 'Subtropical húmedo', 'Ceja de montaña']
df['GI1_4'] = pd.NA
elig = (df['rural'] == 1) & (df['clima'].isin(clima))
df.loc[elig, 'GI1_4'] = 0
df.loc[elig & df['ct_st_d'].gt(0), 'GI1_4'] = 1
df['GI1_4'] = df['GI1_4'].astype('Int8')

ind_nec(df, 'GI1_5', ['ct_cp'])

# GI2
ind_nec(df, 'GI2_2', ['ct_cad'])

# GI3
ind_nec(df, 'GI3_1', ['ct_ce'])
ind_nec(df, 'GI3_2', ['ct_st_me', 'ct_sp_me', 'ct_ri_me', 'ct_rc_me', 
                      'ct_ic_me', 'ct_amp_me'])
ind_nec(df, 'GI3_3', ['ct_ene'])

df['cu'] = df['cu_atyoe'].min()
df['fcr'] = np.where(df['urbano'] == 1, 1, 0.94)
df['area'] = df['areatech'] - (df['areadem'] + df['areari'] + df['arearc']) + \
    0.0001
df['val_act'] = df['area'] * df['cu'] * df['fcr'] * 1.05 * 1.20 * 1.18
df['GI3_4'] = df['val_act'] * 0.01
mask = ((df['finfo'] == 'Sin información') | (df['prioridad'] == '1: Riesgo'))
df.loc[mask, 'GI3_4'] = np.nan
df.drop(columns=['cu', 'fcr', 'area', 'val_act'], inplace=True)

# GI4
ind_nec(df, 'GI4_1', ['ct_sp_r'])
ind_nec(df, 'GI4_3', ['ct_acc'])
ind_nec(df, 'GI4_4', ['ct_amp'])

# GI5
ind_nec(df, 'GI5_1', ['ct_st_ra','ct_st_ae','ct_st_aad'])

##### B. Acceso adecuado a servicios básicos

In [112]:
file = 'data/LE_SSBB_2019.dta'
sb_0 = pd.read_stata(file)
sb_0 = sb_0[['cod_local', 
             'acc_ene',
             'acc_agua',
             'acc_des']]
sb = sb_0.copy()

# Corrigiendo codigos locales
codlocal(sb, 'cod_local', 'codlocal')

In [113]:
# Identificando acceso adecuado
adec = ['1. Red pública', 
        '2. Pilón de uso público',
        '3. Camión-cisterna u otro similar']
sb['agua_adecuado'] = sb['acc_agua'].isin(adec).astype('Int8')

adec = ['1. Red pública',
        '3. Pozo séptico']
sb['sane_adecuado'] = sb['acc_des'].isin(adec).astype('Int8')

sb['GI2_1'] = ((sb['agua_adecuado'] == 1) & 
               (sb['sane_adecuado'] == 1)).astype('Int8')

adec = ['1. Red pública',
        '3. Panel Solar']
sb['GI4_2'] = sb['acc_ene'].isin(adec).astype('Int8')

# Eliminando observaciones sin información de servicios básicos
sb.loc[sb['acc_agua'] == '8. Sin información', 'GI2_1'] = pd.NA
sb.loc[sb['acc_des'] == '7. Sin información', 'GI2_1'] = pd.NA
sb.loc[sb['acc_ene'] == '6. Sin información', 'GI4_2'] = pd.NA

# Seleccionando variables
sb_vf = sb[['codlocal', 'GI2_1', 'GI4_2']]
sb_vf.sort_values('codlocal', inplace=True)
sb_vf = sb_vf.reset_index(drop=True)

##### C. Consolidación

In [114]:
# Uniendo bases
df2 = pd.merge(df, sb_vf, on='codlocal', how='left')

# Seleccionando variables
df = df2[['codlocal', 'brecha', 'prioridad', 'finfo',
             'GI1_1', 'GI1_2', 'GI1_3', 'GI1_4', 'GI1_5',
             'GI2_1', 'GI2_2',
             'GI3_1', 'GI3_2', 'GI3_3', 'GI3_4',
             'GI4_1', 'GI4_2', 'GI4_3', 'GI4_4',
             'GI5_1']]

#### 3.1.2 Indicadores de productos

In [115]:
df = df.copy()

In [116]:
# GI1
iden_nec_prod(df, ['GI1_1', 'GI1_2', 'GI1_3', 'GI1_4', 'GI1_5'], 'GI1')

# GI2
iden_nec_prod(df, ['GI2_1', 'GI2_2'], 'GI2')
df.loc[((df['GI2_1'] == 0) & (df['GI2_2'] == 0)), 'GI2'] = 1

# GI3
iden_nec_prod(df, ['GI3_1', 'GI3_2', 'GI3_3', 'GI3_4'], 'GI3')

# GI4
iden_nec_prod(df, ['GI4_1', 'GI4_2', 'GI4_3', 'GI4_4'], 'GI4')
df.loc[((df['GI4_2'] == 0) & (df['GI4'] == 0)), 'GI4'] = 1

### 3.2 Actividades

In [117]:
# Universo de LLLEE del año
file = 'data/LE_BasePr_2019.dta'
df_0 = pd.read_stata(file)

# Lista de LLEE "existentes"
file = 'data/procesadas/llee_existentes.csv'
exis_0 = pd.read_csv(file, dtype={'codlocal': str})

In [118]:
sfl = df_0.copy()
exis = exis_0.copy()

# Corrigiendo codigos locales
codlocal(sfl, 'cod_local', 'codlocal')

# Identificando LLEE "existentes" y "nuevos"
sfl = pd.merge(sfl, exis,
               how='left',
               on='codlocal',
               indicator='union')
print(sfl['union'].value_counts(dropna=False), '\n')

# Estado del SFL de los predios
freq(sfl, 'disafil')

union
both          48468
left_only      6505
right_only        0
Name: count, dtype: int64 

                            Freq    (%)
disafil                                
No Saneado                18,526  33.7%
NaN                       12,369  22.5%
No Saneado: Valid SUNARP  12,276  22.3%
Saneado                   11,802  21.5%
TOTAL                     54,973   100% 



#### 3.2.1 ET.1 SFL de predios existentes

In [119]:
sfl.loc[sfl['union'] == 'both', 'ET1'] = 0
sfl.loc[(sfl['ET1'] == 0) & (sfl['disafil'] == 'Saneado'), 'ET1'] = 1
sfl.loc[sfl['disafil'].isna(), 'ET1'] = np.nan

freq(sfl, 'ET1')

          Freq    (%)
ET1                  
0.0     27,186  49.5%
 NaN    16,247  29.6%
1.0     11,540  21.0%
 TOTAL  54,973   100% 



#### 3.2.2 ET.2 SFL de predios nuevos

In [120]:
sfl.loc[sfl['union'] == 'left_only', 'ET2'] = 0
sfl.loc[(sfl['ET2'] == 0) & (sfl['disafil'] == 'Saneado'), 'ET2'] = 1
sfl.loc[sfl['disafil'].isna(), 'ET2'] = np.nan

freq(sfl, 'ET2')

          Freq    (%)
ET2                  
 NaN    51,095  92.9%
0.0      3,616   6.6%
1.0        262   0.5%
 TOTAL  54,973   100% 



#### 3.2.3 Consolidación

In [121]:
# Seleccionando variables
sfl = sfl[['codlocal', 'ET1', 'ET2']]

# Consolidando información
df = pd.merge(df, sfl,
              on='codlocal',
              how='left')

### 3.3 Resultados específicos

In [122]:
iden_nec_prod(df, ['GI1', 'GI2', 'GI3_1', 'GI3_2', 'GI4', 'GI5_1', 'ET1'],
              'OE1')
iden_nec_prod(df, ['GI3_3', 'GI3_4'],
              'OE4')

### 3.4 Resultados finales

In [123]:
codlocal(df_0, 'cod_local', 'codlocal')
df = pd.merge(df, df_0[['codlocal', 'matricula']],
              how='left', on='codlocal')

df['RF1'] = 0
df.loc[df['finfo'] == 'Brecha Cerrada', 'RF1'] = 1

df['RF2_matri'] = 0
df.loc[df['RF1'] == 1, 'RF2_matri'] = df['matricula']

df[['matricula', 'RF2_matri']].sum()

C:\Users\Paolo\AppData\Local\Temp\ipykernel_15632\3871407433.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[newvar] = df[oldvar].astype(str).str.zfill(6)


matricula    6242216
RF2_matri     556237
dtype: int64

In [124]:
# Añadiendo año del diagnóstico
sufix = 2019
df.rename(columns={c: f'{c}_{sufix}' for c in df.columns[1:]}, 
             inplace=True)

# Exportando a csv
df.sort_values('codlocal', inplace=True)
df = df.reset_index(drop=True)
df.to_csv('data/procesadas/necint_2019.csv', index=False)

## 04. Año 2020

### 4.1 Productos

#### 4.1.1 Indicadores de subproductos

##### A. Necesidad de intervenciones en infraestructura

In [125]:
file = 'data/LE_BasePr_2020.dta'
df_0 = pd.read_stata(file)
df = df_0.copy()

# Corrigiendo codigos locales
codlocal(df, 'cod_local', 'codlocal')

In [126]:
# GI1
ind_nec(df, 'GI1_1', ['ct_st_d', 'ct_sp_d'])
ind_nec(df, 'GI1_2', ['ct_ri', 'ct_rc'])
ind_nec(df, 'GI1_3', ['ct_ic'])

clima = ['Tropical húmedo', 'Subtropical húmedo', 'Ceja de montaña']
df['GI1_4'] = pd.NA
elig = (df['rural'] == 1) & (df['clima'].isin(clima))
df.loc[elig, 'GI1_4'] = 0
df.loc[elig & df['ct_st_d'].gt(0), 'GI1_4'] = 1
df['GI1_4'] = df['GI1_4'].astype('Int8')

ind_nec(df, 'GI1_5', ['ct_cp'])

# GI2
ind_nec(df, 'GI2_2', ['ct_cad'])

# GI3
ind_nec(df, 'GI3_1', ['ct_ce'])
ind_nec(df, 'GI3_2', ['ct_st_me', 'ct_sp_me', 'ct_ri_me', 'ct_rc_me', 
                      'ct_ic_me', 'ct_amp_me'])
ind_nec(df, 'GI3_3', ['ct_ene'])

df['cu'] = df['cu_atyoe'].min()
df['fcr'] = np.where(df['urbano'] == 1, 1, 0.94)
df['area'] = df['areatech'] - (df['areadem'] + df['areari'] + df['arearc']) + \
    0.0001
df['val_act'] = df['area'] * df['cu'] * df['fcr'] * 1.05 * 1.20 * 1.18
df['GI3_4'] = df['val_act'] * 0.01
mask = ((df['finfo'] == 'Sin información') | (df['prioridad'] == '1: Riesgo'))
df.loc[mask, 'GI3_4'] = np.nan
df.drop(columns=['cu', 'fcr', 'area', 'val_act'], inplace=True)

# GI4
ind_nec(df, 'GI4_1', ['ct_sp_r'])
ind_nec(df, 'GI4_3', ['ct_acc'])
ind_nec(df, 'GI4_4', ['ct_amp'])

# GI5
ind_nec(df, 'GI5_1', ['ct_st_ra','ct_st_ae','ct_st_aad'])

##### B. Acceso adecuado a servicios básicos

In [127]:
file = 'data/LE_SSBB_2020.dta'
sb_0 = pd.read_stata(file)
sb_0 = sb_0[['cod_local', 
             'acc_ene',
             'acc_agua',
             'acc_des']]
sb = sb_0.copy()

# Corrigiendo codigos locales
codlocal(sb, 'cod_local', 'codlocal')

In [128]:
# Identificando acceso adecuado
adec = ['1. Red pública', 
        '2. Pilón de uso público',
        '3. Camión-cisterna u otro similar']
sb['agua_adecuado'] = sb['acc_agua'].isin(adec).astype('Int8')

adec = ['1. Red pública',
        '3. Pozo séptico']
sb['sane_adecuado'] = sb['acc_des'].isin(adec).astype('Int8')

sb['GI2_1'] = ((sb['agua_adecuado'] == 1) & 
               (sb['sane_adecuado'] == 1)).astype('Int8')

adec = ['1. Red pública',
        '3. Panel solar']
sb['GI4_2'] = sb['acc_ene'].isin(adec).astype('Int8')

# Eliminando observaciones sin información de servicios básicos
sb.loc[sb['acc_agua'] == '8. Sin información', 'GI2_1'] = pd.NA
sb.loc[sb['acc_des'] == '7. Sin información', 'GI2_1'] = pd.NA
sb.loc[sb['acc_ene'] == '6. Sin información', 'GI4_2'] = pd.NA

# Seleccionando variables
sb_vf = sb[['codlocal', 'GI2_1', 'GI4_2']]
sb_vf.sort_values('codlocal', inplace=True)
sb_vf = sb_vf.reset_index(drop=True)

##### C. Consolidación

In [129]:
# Uniendo bases
df2 = pd.merge(df, sb_vf, on='codlocal', how='left')

# Seleccionando variables
df = df2[['codlocal', 'brecha', 'prioridad', 'finfo',
             'GI1_1', 'GI1_2', 'GI1_3', 'GI1_4', 'GI1_5',
             'GI2_1', 'GI2_2',
             'GI3_1', 'GI3_2', 'GI3_3', 'GI3_4',
             'GI4_1', 'GI4_2', 'GI4_3', 'GI4_4',
             'GI5_1']]

#### 4.1.2 Indicadores de productos

In [130]:
df = df.copy()

In [131]:
# GI1
iden_nec_prod(df, ['GI1_1', 'GI1_2', 'GI1_3', 'GI1_4', 'GI1_5'], 'GI1')

# GI2
iden_nec_prod(df, ['GI2_1', 'GI2_2'], 'GI2')
df.loc[((df['GI2_1'] == 0) & (df['GI2_2'] == 0)), 'GI2'] = 1

# GI3
iden_nec_prod(df, ['GI3_1', 'GI3_2', 'GI3_3', 'GI3_4'], 'GI3')

# GI4
iden_nec_prod(df, ['GI4_1', 'GI4_2', 'GI4_3', 'GI4_4'], 'GI4')
df.loc[((df['GI4_2'] == 0) & (df['GI4'] == 0)), 'GI4'] = 1

### 4.2 Actividades

In [132]:
# Universo de LLLEE del año
file = 'data/LE_BasePr_2020.dta'
df_0 = pd.read_stata(file)

# Lista de LLEE "existentes"
file = 'data/procesadas/llee_existentes.csv'
exis_0 = pd.read_csv(file, dtype={'codlocal': str})

In [133]:
sfl = df_0.copy()
exis = exis_0.copy()

# Corrigiendo codigos locales
codlocal(sfl, 'cod_local', 'codlocal')

# Identificando LLEE "existentes" y "nuevos"
sfl = pd.merge(sfl, exis,
               how='left',
               on='codlocal',
               indicator='union')
print(sfl['union'].value_counts(dropna=False), '\n')

# Estado del SFL de los predios
freq(sfl, 'disafil')

union
both          48358
left_only      6700
right_only        0
Name: count, dtype: int64 

               Freq    (%)
disafil                   
NaN          30,455  55.3%
No Saneado   12,089  22.0%
Saneado      11,734  21.3%
En revisión     780   1.4%
TOTAL        55,058   100% 



#### 4.2.1 ET.1 SFL de predios existentes

In [134]:
sfl.loc[sfl['union'] == 'both', 'ET1'] = 0
sfl.loc[(sfl['ET1'] == 0) & (sfl['disafil'] == 'Saneado'), 'ET1'] = 1
sfl.loc[sfl['disafil'].isna(), 'ET1'] = np.nan

freq(sfl, 'ET1')

          Freq    (%)
ET1                  
 NaN    30,867  56.1%
0.0     12,734  23.1%
1.0     11,457  20.8%
 TOTAL  55,058   100% 



#### 4.2.2 ET.2 SFL de predios nuevos

In [135]:
sfl.loc[sfl['union'] == 'left_only', 'ET2'] = 0
sfl.loc[(sfl['ET2'] == 0) & (sfl['disafil'] == 'Saneado'), 'ET2'] = 1
sfl.loc[sfl['disafil'].isna(), 'ET2'] = np.nan

freq(sfl, 'ET2')

          Freq    (%)
ET2                  
 NaN    54,646  99.3%
1.0        277   0.5%
0.0        135   0.2%
 TOTAL  55,058   100% 



#### 4.2.3 Consolidación

In [136]:
# Seleccionando variables
sfl = sfl[['codlocal', 'ET1', 'ET2']]

# Consolidando información
df = pd.merge(df, sfl,
              on='codlocal',
              how='left')

### 4.3 Resultados específicos

In [137]:
iden_nec_prod(df, ['GI1', 'GI2', 'GI3_1', 'GI3_2', 'GI4', 'GI5_1', 'ET1'],
              'OE1')
iden_nec_prod(df, ['GI3_3', 'GI3_4'],
              'OE4')

### 4.4 Resultados finales

In [138]:
codlocal(df_0, 'cod_local', 'codlocal')
df = pd.merge(df, df_0[['codlocal', 'matricula']],
              how='left', on='codlocal')

df['RF1'] = 0
df.loc[df['finfo'] == 'Brecha Cerrada', 'RF1'] = 1

df['RF2_matri'] = 0
df.loc[df['RF1'] == 1, 'RF2_matri'] = df['matricula']

df[['matricula', 'RF2_matri']].sum()

C:\Users\Paolo\AppData\Local\Temp\ipykernel_15632\3871407433.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[newvar] = df[oldvar].astype(str).str.zfill(6)


matricula    6585668
RF2_matri     733109
dtype: int64

In [139]:
# Añadiendo año del diagnóstico
sufix = 2020
df.rename(columns={c: f'{c}_{sufix}' for c in df.columns[1:]}, 
             inplace=True)

# Exportando a csv
df.sort_values('codlocal', inplace=True)
df = df.reset_index(drop=True)
df.to_csv('data/procesadas/necint_2020.csv', index=False)

## 05. Año 2021

### 5.1 Productos

#### 5.1.1 Indicadores de subproductos

##### A. Necesidad de intervenciones en infraestructura

In [140]:
file = 'data/LE_BasePr_2021.dta'
df_0 = pd.read_stata(file)
df = df_0.copy()

# Corrigiendo codigos locales
codlocal(df, 'cod_local', 'codlocal')

C:\Users\Paolo\AppData\Local\Temp\ipykernel_15632\4045229493.py:2: UnicodeWarning: 
One or more strings in the dta file could not be decoded using utf-8, and
so the fallback encoding of latin-1 is being used.  This can happen when a file
has been incorrectly encoded by Stata or some other software. You should verify
the string values returned are correct.
  df_0 = pd.read_stata(file)


In [141]:
# GI1
ind_nec(df, 'GI1_1', ['ct_st_d', 'ct_sp_d'])
ind_nec(df, 'GI1_2', ['ct_ri', 'ct_rc'])
ind_nec(df, 'GI1_3', ['ct_ic'])

clima = ['Tropical húmedo', 'Subtropical húmedo', 'Ceja de montaña']
df['GI1_4'] = pd.NA
elig = (df['rural'] == 1) & (df['clima'].isin(clima))
df.loc[elig, 'GI1_4'] = 0
df.loc[elig & df['ct_st_d'].gt(0), 'GI1_4'] = 1
df['GI1_4'] = df['GI1_4'].astype('Int8')

ind_nec(df, 'GI1_5', ['ct_cp'])

# GI2
ind_nec(df, 'GI2_2', ['ct_cad'])

# GI3
ind_nec(df, 'GI3_1', ['ct_ce'])
ind_nec(df, 'GI3_2', ['ct_st_me', 'ct_sp_me', 'ct_ri_me', 'ct_rc_me', 
                      'ct_ic_me', 'ct_amp_me'])
ind_nec(df, 'GI3_3', ['ct_ene'])

df['cu'] = df['cu_atyoe'].min()
df['fcr'] = np.where(df['urbano'] == 1, 1, 0.94)
df['area'] = df['areatech'] - (df['areadem'] + df['areari'] + df['arearc']) + \
    0.0001
df['val_act'] = df['area'] * df['cu'] * df['fcr'] * 1.05 * 1.20 * 1.18
df['GI3_4'] = df['val_act'] * 0.01
mask = ((df['finfo'] == 'Sin información') | (df['prioridad'] == '1: Riesgo'))
df.loc[mask, 'GI3_4'] = np.nan
df.drop(columns=['cu', 'fcr', 'area', 'val_act'], inplace=True)

# GI4
ind_nec(df, 'GI4_1', ['ct_sp_r'])
ind_nec(df, 'GI4_3', ['ct_acc'])
ind_nec(df, 'GI4_4', ['ct_amp'])

# GI5
ind_nec(df, 'GI5_1', ['ct_st_ra','ct_st_ae','ct_st_aad'])

##### B. Acceso adecuado a servicios básicos

In [142]:
file = 'data/LE_SSBB_2021.dta'
sb_0 = pd.read_stata(file)
sb_0 = sb_0[['cod_local', 'acc_ene', 'acc_agua', 'acc_des']]
sb = sb_0.copy()

# Corrigiendo codigos locales
codlocal(sb, 'cod_local', 'codlocal')

In [143]:
# Identificando acceso adecuado
adec = ['1. Red pública (agua potable)', 
        '2. Pilón de uso público (agua potable)',
        '3. Camión cisterna u otro similar']
sb['agua_adecuado'] = sb['acc_agua'].isin(adec).astype('Int8')

adec = ['1. Red pública',
        '3. Tanque séptico',
        '6. Biodigestor']
sb['sane_adecuado'] = sb['acc_des'].isin(adec).astype('Int8')

sb['GI2_1'] = ((sb['agua_adecuado'] == 1) & 
               (sb['sane_adecuado'] == 1)).astype('Int8')

adec = ['1. Red pública (De una empresa distribuidora de energía eléctrica)',
        '5. Panel solar',
        '6. Energía eólica']
sb['GI4_2'] = sb['acc_ene'].isin(adec).astype('Int8')

# Eliminando observaciones sin información de servicios básicos
sb.loc[sb['acc_agua'] == '8. Sin información', 'GI2_1'] = pd.NA
sb.loc[sb['acc_des'] == '9. Sin información', 'GI2_1'] = pd.NA
sb.loc[sb['acc_ene'] == '9. Sin información', 'GI4_2'] = pd.NA

# Seleccionando variables
sb_vf = sb[['codlocal', 'GI2_1', 'GI4_2']]
sb_vf.sort_values('codlocal', inplace=True)
sb_vf = sb_vf.reset_index(drop=True)

##### C. Consolidación

In [144]:
# Uniendo bases
df2 = pd.merge(df, sb_vf, on='codlocal', how='left')

# Seleccionando variables
df = df2[['codlocal', 'brecha', 'prioridad', 'finfo',
             'GI1_1', 'GI1_2', 'GI1_3', 'GI1_4', 'GI1_5',
             'GI2_1', 'GI2_2',
             'GI3_1', 'GI3_2', 'GI3_3', 'GI3_4',
             'GI4_1', 'GI4_2', 'GI4_3', 'GI4_4',
             'GI5_1']]

#### 5.1.2 Indicadores de productos

In [145]:
df = df.copy()

In [146]:
# GI1
iden_nec_prod(df, ['GI1_1', 'GI1_2', 'GI1_3', 'GI1_4', 'GI1_5'], 'GI1')

# GI2
iden_nec_prod(df, ['GI2_1', 'GI2_2'], 'GI2')
df.loc[((df['GI2_1'] == 0) & (df['GI2_2'] == 0)), 'GI2'] = 1

# GI3
iden_nec_prod(df, ['GI3_1', 'GI3_2', 'GI3_3', 'GI3_4'], 'GI3')

# GI4
iden_nec_prod(df, ['GI4_1', 'GI4_2', 'GI4_3', 'GI4_4'], 'GI4')
df.loc[((df['GI4_2'] == 0) & (df['GI4'] == 0)), 'GI4'] = 1

### 5.2 Actividades

In [147]:
# Universo de LLLEE del año
file = 'data/LE_BasePr_2021.dta'
df_0 = pd.read_stata(file)

# Lista de LLEE "existentes"
file = 'data/procesadas/llee_existentes.csv'
exis_0 = pd.read_csv(file, dtype={'codlocal': str})

C:\Users\Paolo\AppData\Local\Temp\ipykernel_15632\2137641370.py:3: UnicodeWarning: 
One or more strings in the dta file could not be decoded using utf-8, and
so the fallback encoding of latin-1 is being used.  This can happen when a file
has been incorrectly encoded by Stata or some other software. You should verify
the string values returned are correct.
  df_0 = pd.read_stata(file)


In [148]:
sfl = df_0.copy()
exis = exis_0.copy()

# Corrigiendo codigos locales
codlocal(sfl, 'cod_local', 'codlocal')

# Identificando LLEE "existentes" y "nuevos"
sfl = pd.merge(sfl, exis,
               how='left',
               on='codlocal',
               indicator='union')
print(sfl['union'].value_counts(dropna=False), '\n')

# Estado del SFL de los predios
sfl['disafil'] = sfl['disafil'].cat.add_categories('No saneado')
sfl['disafil'] = sfl['disafil'].fillna('No saneado')
freq(sfl, 'disafil')

union
both          48331
left_only      6880
right_only        0
Name: count, dtype: int64 

                 Freq    (%)
disafil                     
No registrado  26,415  47.8%
Saneado        18,095  32.8%
No saneado     10,701  19.4%
TOTAL          55,211   100% 



#### 5.2.1 ET.1 SFL de predios existentes

In [149]:
sfl.loc[sfl['union'] == 'both', 'ET1'] = 0
sfl.loc[(sfl['ET1'] == 0) & (sfl['disafil'] == 'Saneado'), 'ET1'] = 1
sfl.loc[sfl['disafil'] == 'No registrado', 'ET1'] = np.nan

freq(sfl, 'ET1')

          Freq    (%)
ET1                  
 NaN    27,367  49.6%
1.0     17,252  31.2%
0.0     10,592  19.2%
 TOTAL  55,211   100% 



#### 5.2.2 ET.2 SFL de predios nuevos

In [150]:
sfl.loc[sfl['union'] == 'left_only', 'ET2'] = 0
sfl.loc[(sfl['ET2'] == 0) & (sfl['disafil'] == 'Saneado'), 'ET2'] = 1
sfl.loc[sfl['disafil'] == 'No registrado', 'ET2'] = np.nan

freq(sfl, 'ET2')

          Freq    (%)
ET2                  
 NaN    54,259  98.3%
1.0        843   1.5%
0.0        109   0.2%
 TOTAL  55,211   100% 



#### 5.2.3 Consolidación

In [151]:
# Seleccionando variables
sfl = sfl[['codlocal', 'ET1', 'ET2']]

# Consolidando información
df = pd.merge(df, sfl,
              on='codlocal',
              how='left')

### 5.3 Resultados específicos

In [152]:
iden_nec_prod(df, ['GI1', 'GI2', 'GI3_1', 'GI3_2', 'GI4', 'GI5_1', 'ET1'],
              'OE1')
iden_nec_prod(df, ['GI3_3', 'GI3_4'],
              'OE4')

### 5.4 Resultados finales

In [153]:
codlocal(df_0, 'cod_local', 'codlocal')
df = pd.merge(df, df_0[['codlocal', 'matricula']],
              how='left', on='codlocal')

df['RF1'] = 0
df.loc[df['finfo'] == 'Brecha Cerrada', 'RF1'] = 1

df['RF2_matri'] = 0
df.loc[df['RF1'] == 1, 'RF2_matri'] = df['matricula']

df[['matricula', 'RF2_matri']].sum()

C:\Users\Paolo\AppData\Local\Temp\ipykernel_15632\3871407433.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[newvar] = df[oldvar].astype(str).str.zfill(6)


matricula    6743853
RF2_matri     926322
dtype: int64

In [154]:
# Añadiendo año del diagnóstico
sufix = 2021
df.rename(columns={c: f'{c}_{sufix}' for c in df.columns[1:]}, 
             inplace=True)

# Exportando a csv
df.sort_values('codlocal', inplace=True)
df = df.reset_index(drop=True)
df.to_csv('data/procesadas/necint_2021.csv', index=False)

## 06. Año 2022

### 6.1 Productos

#### 6.1.1 Indicadores de subproductos

##### A. Necesidad de intervenciones en infraestructura

In [155]:
file = 'data/LE_BasePr_2022.dta'
df_0 = pd.read_stata(file)
df = df_0.copy()

# Corrigiendo codigos locales
codlocal(df, 'cod_local', 'codlocal')

In [156]:
# GI1
ind_nec(df, 'GI1_1', ['ct_st_d', 'ct_sp_d'])
ind_nec(df, 'GI1_2', ['ct_ri', 'ct_rc'])
ind_nec(df, 'GI1_3', ['ct_ic'])

clima = ['Tropical húmedo', 'Subtropical húmedo', 'Ceja de montaña']
df['GI1_4'] = pd.NA
elig = (df['rural'] == 1) & (df['clima'].isin(clima))
df.loc[elig, 'GI1_4'] = 0
df.loc[elig & df['ct_st_d'].gt(0), 'GI1_4'] = 1
df['GI1_4'] = df['GI1_4'].astype('Int8')

ind_nec(df, 'GI1_5', ['ct_cp'])

# GI2
ind_nec(df, 'GI2_2', ['ct_cad'])

# GI3
ind_nec(df, 'GI3_1', ['ct_ce'])
ind_nec(df, 'GI3_2', ['ct_st_me', 'ct_sp_me', 'ct_ri_me', 'ct_rc_me', 
                      'ct_ic_me', 'ct_amp_me'])
ind_nec(df, 'GI3_3', ['ct_ene'])

df['cu'] = df['cu_atyoe'].min()
df['fcr'] = np.where(df['urbano'] == 1, 1, 0.94)
df['area'] = df['areatech'] - (df['areadem'] + df['areari'] + df['arearc']) + \
    0.0001
df['val_act'] = df['area'] * df['cu'] * df['fcr'] * 1.05 * 1.20 * 1.18
df['GI3_4'] = df['val_act'] * 0.01
mask = ((df['finfo'] == 'Sin información') | (df['prioridad'] == '1: Riesgo'))
df.loc[mask, 'GI3_4'] = np.nan
df.drop(columns=['cu', 'fcr', 'area', 'val_act'], inplace=True)

# GI4
ind_nec(df, 'GI4_1', ['ct_sp_r'])
ind_nec(df, 'GI4_3', ['ct_acc'])
ind_nec(df, 'GI4_4', ['ct_amp'])

# GI5
ind_nec(df, 'GI5_1', ['ct_st_ra','ct_st_ae','ct_st_aad'])

##### B. Acceso adecuado a servicios básicos

In [157]:
file = 'data/LE_SSBB_2022.dta'
sb_0 = pd.read_stata(file)
sb_0 = sb_0[['cod_local', 'acc_ene', 'acc_agua', 'acc_des']]
sb = sb_0.copy()

# Corrigiendo codigos locales
codlocal(sb, 'cod_local', 'codlocal')

In [158]:
# Identificando acceso adecuado
adec = ['1. Red pública (agua potable)', 
        '2. Pilón de uso público (agua potable)',
        '3. Camión cisterna u otro similar']
sb['agua_adecuado'] = sb['acc_agua'].isin(adec).astype('Int8')

adec = ['01. Red pública',
        '03. Tanque séptico',
        '06. Biodigestor']
sb['sane_adecuado'] = sb['acc_des'].isin(adec).astype('Int8')

sb['GI2_1'] = ((sb['agua_adecuado'] == 1) & 
               (sb['sane_adecuado'] == 1)).astype('Int8')

adec = ['1. Red pública (de una empresa distribuidora de energía eléctrica)',
        '5. Panel solar',
        '6. Energía eólica']
sb['GI4_2'] = sb['acc_ene'].isin(adec).astype('Int8')

# Eliminando observaciones sin información de servicios básicos
sb.loc[sb['acc_agua'] == '8. Sin información', 'GI2_1'] = pd.NA
sb.loc[sb['acc_des'] == '10. Sin información', 'GI2_1'] = pd.NA
sb.loc[sb['acc_ene'] == '9. Sin información', 'GI4_2'] = pd.NA

# Seleccionando variables
sb_vf = sb[['codlocal', 'GI2_1', 'GI4_2']]
sb_vf.sort_values('codlocal', inplace=True)
sb_vf = sb_vf.reset_index(drop=True)

##### C. Consolidación

In [159]:
# Uniendo bases
df2 = pd.merge(df, sb_vf, on='codlocal', how='left')

# Seleccionando variables
df = df2[['codlocal', 'brecha', 'prioridad', 'finfo',
             'GI1_1', 'GI1_2', 'GI1_3', 'GI1_4', 'GI1_5',
             'GI2_1', 'GI2_2',
             'GI3_1', 'GI3_2', 'GI3_3', 'GI3_4',
             'GI4_1', 'GI4_2', 'GI4_3', 'GI4_4',
             'GI5_1']]

#### 6.1.2 Indicadores de productos

In [160]:
df = df.copy()

In [161]:
# GI1
iden_nec_prod(df, ['GI1_1', 'GI1_2', 'GI1_3', 'GI1_4', 'GI1_5'], 'GI1')

# GI2
iden_nec_prod(df, ['GI2_1', 'GI2_2'], 'GI2')
df.loc[((df['GI2_1'] == 0) & (df['GI2_2'] == 0)), 'GI2'] = 1

# GI3
iden_nec_prod(df, ['GI3_1', 'GI3_2', 'GI3_3', 'GI3_4'], 'GI3')

# GI4
iden_nec_prod(df, ['GI4_1', 'GI4_2', 'GI4_3', 'GI4_4'], 'GI4')
df.loc[((df['GI4_2'] == 0) & (df['GI4'] == 0)), 'GI4'] = 1

### 6.2 Actividades

In [162]:
# Universo de LLLEE del año
file = 'data/LE_BasePr_2022.dta'
df_0 = pd.read_stata(file)

# Lista de LLEE "existentes"
file = 'data/procesadas/llee_existentes.csv'
exis_0 = pd.read_csv(file, dtype={'codlocal': str})

In [163]:
sfl = df_0.copy()
exis = exis_0.copy()

# Corrigiendo codigos locales
codlocal(sfl, 'cod_local', 'codlocal')

# Identificando LLEE "existentes" y "nuevos"
sfl = pd.merge(sfl, exis,
               how='left',
               on='codlocal',
               indicator='union')
print(sfl['union'].value_counts(dropna=False), '\n')

# Estado del SFL de los predios
freq(sfl, 'disafil')

union
both          48212
left_only      7092
right_only        0
Name: count, dtype: int64 

                 Freq    (%)
disafil                     
No registrado  25,396  45.9%
Saneado        18,671  33.8%
No saneado     11,209  20.3%
NaN                28   0.1%
TOTAL          55,304   100% 



#### 6.2.1 ET.1 SFL de predios existentes

In [164]:
sfl.loc[sfl['union'] == 'both', 'ET1'] = 0
sfl.loc[(sfl['ET1'] == 0) & (sfl['disafil'] == 'Saneado'), 'ET1'] = 1
sfl.loc[(sfl['disafil'] == 'No registrado') | (sfl['disafil'].isna()), 
        'ET1'] = np.nan

freq(sfl, 'ET1')

          Freq    (%)
ET1                  
 NaN    26,649  48.2%
1.0     17,657  31.9%
0.0     10,998  19.9%
 TOTAL  55,304   100% 



#### 6.2.2 ET.2 SFL de predios nuevos

In [165]:
sfl.loc[sfl['union'] == 'left_only', 'ET2'] = 0
sfl.loc[(sfl['ET2'] == 0) & (sfl['disafil'] == 'Saneado'), 'ET2'] = 1
sfl.loc[(sfl['disafil'] == 'No registrado') | (sfl['disafil'].isna()), 
        'ET2'] = np.nan

freq(sfl, 'ET2')

          Freq    (%)
ET2                  
 NaN    54,079  97.8%
1.0      1,014   1.8%
0.0        211   0.4%
 TOTAL  55,304   100% 



#### 6.2.3 Consolidación

In [166]:
# Seleccionando variables
sfl = sfl[['codlocal', 'ET1', 'ET2']]

# Consolidando información
df = pd.merge(df, sfl,
              on='codlocal',
              how='left')

### 6.3 Resultados específicos

In [167]:
iden_nec_prod(df, ['GI1', 'GI2', 'GI3_1', 'GI3_2', 'GI4', 'GI5_1', 'ET1'],
              'OE1')
iden_nec_prod(df, ['GI3_3', 'GI3_4'],
              'OE4')

### 6.4 Resultados finales

In [168]:
codlocal(df_0, 'cod_local', 'codlocal')
df = pd.merge(df, df_0[['codlocal', 'matricula']],
              how='left', on='codlocal')

df['RF1'] = 0
df.loc[df['finfo'] == 'Brecha Cerrada', 'RF1'] = 1

df['RF2_matri'] = 0
df.loc[df['RF1'] == 1, 'RF2_matri'] = df['matricula']

df[['matricula', 'RF2_matri']].sum()

C:\Users\Paolo\AppData\Local\Temp\ipykernel_15632\3871407433.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[newvar] = df[oldvar].astype(str).str.zfill(6)


matricula    6569996
RF2_matri     800611
dtype: int64

In [169]:
# Añadiendo año del diagnóstico
sufix = 2022
df.rename(columns={c: f'{c}_{sufix}' for c in df.columns[1:]}, 
             inplace=True)

# Exportando a csv
df.sort_values('codlocal', inplace=True)
df = df.reset_index(drop=True)
df.to_csv('data/procesadas/necint_2022.csv', index=False)
# df

## 07. Año 2023

### 7.1 Productos

#### 7.1.1 Indicadores de subproductos

##### A. Necesidad de intervenciones en infraestructura

In [170]:
file = 'data/LE_BasePr_2023.dta'
df_0 = pd.read_stata(file)
df = df_0.copy()

# Corrigiendo codigos locales
codlocal(df, 'cod_local', 'codlocal')

In [171]:
# GI1
ind_nec(df, 'GI1_1', ['ct_st_d', 'ct_sp_d'])
ind_nec(df, 'GI1_2', ['ct_ri', 'ct_rc'])
ind_nec(df, 'GI1_3', ['ct_ic'])

clima = ['Tropical húmedo', 'Subtropical húmedo', 'Ceja de montaña']
df['GI1_4'] = pd.NA
elig = (df['rural'] == 1) & (df['clima'].isin(clima))
df.loc[elig, 'GI1_4'] = 0
df.loc[elig & df['ct_st_d'].gt(0), 'GI1_4'] = 1
df['GI1_4'] = df['GI1_4'].astype('Int8')

ind_nec(df, 'GI1_5', ['ct_cp'])

# GI2
ind_nec(df, 'GI2_2', ['ct_cad'])

# GI3
ind_nec(df, 'GI3_1', ['ct_ce'])
ind_nec(df, 'GI3_2', ['ct_st_me', 'ct_sp_me', 'ct_ri_me', 'ct_rc_me', 
                      'ct_ic_me', 'ct_amp_me'])
ind_nec(df, 'GI3_3', ['ct_ene'])

df['cu'] = df['cu_atyoe'].min()
df['fcr'] = np.where(df['urbano'] == 1, 1, 0.94)
df['area'] = df['areatech'] - (df['areadem'] + df['areari'] + df['arearc']) + \
    0.0001
df['val_act'] = df['area'] * df['cu'] * df['fcr'] * 1.05 * 1.20 * 1.18
df['GI3_4'] = df['val_act'] * 0.01
mask = ((df['finfo'] == 'Sin información') | (df['prioridad'] == '1: Riesgo'))
df.loc[mask, 'GI3_4'] = np.nan
df.drop(columns=['cu', 'fcr', 'area', 'val_act'], inplace=True)

# GI4
ind_nec(df, 'GI4_1', ['ct_sp_r'])
ind_nec(df, 'GI4_3', ['ct_acc'])
ind_nec(df, 'GI4_4', ['ct_amp'])

# GI5
ind_nec(df, 'GI5_1', ['ct_st_ra','ct_st_ae','ct_st_aad'])

##### B. Acceso adecuado a servicios básicos

In [172]:
file = 'data/LE_SSBB_2023.xlsx'
sb_0 = pd.read_excel(file, sheet_name='BD')
sb_0 = sb_0[['Código Local', 
             'Procedencia de abastecimiento - Agua',
             'Tipo de conexión - Alcantarillado Sanitario',
             'Procedencia de abastecimiento - Energía eléctrica']]
sb = sb_0.copy()

# Corrigiendo codigos locales
codlocal(sb, 'Código Local', 'codlocal')

In [173]:
# Identificando acceso adecuado
adec = ['1. Red pública (agua potable)', 
        '2. Pilón de uso público (agua potable)',
        '3. Camión cisterna u otro similar']
sb['agua_adecuado'] = sb['Procedencia de abastecimiento - Agua'].isin(adec). \
    astype('Int8')

adec = ['1. Red pública',
        '3. Tanque séptico',
        '4. Biodigestor',
        '6. Unidades Básicas de Saneamiento de compostera (U.B.S -C)']
sb['sane_adecuado'] = sb['Tipo de conexión - Alcantarillado Sanitario']. \
    isin(adec).astype('Int8')

sb['GI2_1'] = ((sb['agua_adecuado'] == 1) & 
               (sb['sane_adecuado'] == 1)).astype('Int8')

adec = ['1. Red pública (De una empresa distribuidora de energía eléctrica)',
        '5. Panel solar',
        '6. Energía Eólica']
sb['GI4_2'] = sb['Procedencia de abastecimiento - Energía eléctrica']. \
    isin(adec).astype('Int8')

# Eliminando observaciones sin información de servicios básicos
sb.loc[sb['Procedencia de abastecimiento - Agua'] == '9. Sin información',
       'GI2_1'] = pd.NA
sb.loc[sb['Tipo de conexión - Alcantarillado Sanitario'] == \
       '9. Sin información', 'GI2_1'] = pd.NA
sb.loc[sb['Procedencia de abastecimiento - Energía eléctrica'] == \
       '9. Sin información', 'GI4_2'] = pd.NA

# Seleccionando variables
sb_vf = sb[['codlocal', 'GI2_1', 'GI4_2']]
sb_vf.sort_values('codlocal', inplace=True)
sb_vf = sb_vf.reset_index(drop=True)

##### C. Consolidación

In [174]:
# Uniendo bases
df2 = pd.merge(df, sb_vf, on='codlocal', how='left')

# Seleccionando variables
df = df2[['codlocal', 'brecha', 'prioridad', 'finfo',
             'GI1_1', 'GI1_2', 'GI1_3', 'GI1_4', 'GI1_5',
             'GI2_1', 'GI2_2',
             'GI3_1', 'GI3_2', 'GI3_3', 'GI3_4',
             'GI4_1', 'GI4_2', 'GI4_3', 'GI4_4',
             'GI5_1']]

#### 7.1.2 Indicadores de productos

In [175]:
df = df.copy()

In [176]:
# GI1
iden_nec_prod(df, ['GI1_1', 'GI1_2', 'GI1_3', 'GI1_4', 'GI1_5'], 'GI1')

# GI2
iden_nec_prod(df, ['GI2_1', 'GI2_2'], 'GI2')
df.loc[((df['GI2_1'] == 0) & (df['GI2_2'] == 0)), 'GI2'] = 1

# GI3
iden_nec_prod(df, ['GI3_1', 'GI3_2', 'GI3_3', 'GI3_4'], 'GI3')

# GI4
iden_nec_prod(df, ['GI4_1', 'GI4_2', 'GI4_3', 'GI4_4'], 'GI4')
df.loc[((df['GI4_2'] == 0) & (df['GI4'] == 0)), 'GI4'] = 1

### 7.2 Actividades

In [177]:
# Universo de LLLEE del año
file = 'data/LE_BasePr_2023.dta'
df_0 = pd.read_stata(file)

# Lista de LLEE "existentes"
file = 'data/procesadas/llee_existentes.csv'
exis_0 = pd.read_csv(file, dtype={'codlocal': str})

In [178]:
sfl = df_0.copy()
exis = exis_0.copy()

# Corrigiendo codigos locales
codlocal(sfl, 'cod_local', 'codlocal')

# Identificando LLEE "existentes" y "nuevos"
sfl = pd.merge(sfl, exis,
               how='left',
               on='codlocal',
               indicator='union')
print(sfl['union'].value_counts(dropna=False), '\n')

# Estado del SFL de los predios
freq(sfl, 'disafil')

union
both          48060
left_only      7298
right_only        0
Name: count, dtype: int64 

                 Freq    (%)
disafil                     
No registrado  23,133  41.8%
Saneado        19,590  35.4%
No saneado     12,635  22.8%
TOTAL          55,358   100% 



#### 7.2.1 ET.1 SFL de predios existentes

In [179]:
sfl.loc[sfl['union'] == 'both', 'ET1'] = 0
sfl.loc[(sfl['ET1'] == 0) & (sfl['disafil'] == 'Saneado'), 'ET1'] = 1
sfl.loc[sfl['disafil'] == 'No registrado', 'ET1'] = np.nan

freq(sfl, 'ET1')

          Freq    (%)
ET1                  
 NaN    24,850  44.9%
1.0     18,444  33.3%
0.0     12,064  21.8%
 TOTAL  55,358   100% 



#### 7.2.2 ET.2 SFL de predios nuevos

In [180]:
sfl.loc[sfl['union'] == 'left_only', 'ET2'] = 0
sfl.loc[(sfl['ET2'] == 0) & (sfl['disafil'] == 'Saneado'), 'ET2'] = 1
sfl.loc[sfl['disafil'] == 'No registrado', 'ET2'] = np.nan

freq(sfl, 'ET2')

          Freq    (%)
ET2                  
 NaN    53,641  96.9%
1.0      1,146   2.1%
0.0        571   1.0%
 TOTAL  55,358   100% 



#### 7.2.3 Consolidación

In [181]:
# Seleccionando variables
sfl = sfl[['codlocal', 'ET1', 'ET2']]

# Consolidando información
df = pd.merge(df, sfl,
              on='codlocal',
              how='left')

### 7.3 Resultados específicos

In [182]:
iden_nec_prod(df, ['GI1', 'GI2', 'GI3_1', 'GI3_2', 'GI4', 'GI5_1', 'ET1'],
              'OE1')
iden_nec_prod(df, ['GI3_3', 'GI3_4'],
              'OE4')

### 7.4 Resultados finales

In [183]:
codlocal(df_0, 'cod_local', 'codlocal')
df = pd.merge(df, df_0[['codlocal', 'matricula']],
              how='left', on='codlocal')

df['RF1'] = 0
df.loc[df['finfo'] == 'Brecha Cerrada', 'RF1'] = 1

df['RF2_matri'] = 0
df.loc[df['RF1'] == 1, 'RF2_matri'] = df['matricula']

df[['matricula', 'RF2_matri']].sum()

matricula    6529623
RF2_matri     799894
dtype: int64

In [184]:
# Añadiendo año del diagnóstico
sufix = 2023
df.rename(columns={c: f'{c}_{sufix}' for c in df.columns[1:]}, 
             inplace=True)

# Exportando a csv
df.sort_values('codlocal', inplace=True)
df = df.reset_index(drop=True)
df.to_csv('data/procesadas/necint_2023.csv', index=False)
# df

## 08. Año 2024

### 8.1 Productos

#### 8.1.1 Indicadores de subproductos

##### A. Necesidad de intervenciones en infraestructura

In [185]:
file = 'data/LE_BasePr_2024.dta'
df_0 = pd.read_stata(file)
df = df_0.copy()

# Corrigiendo codigos locales
codlocal(df, 'cod_local', 'codlocal')

In [186]:
# GI1
ind_nec(df, 'GI1_1', ['ct_st_d', 'ct_sp_d'])
ind_nec(df, 'GI1_2', ['ct_ri', 'ct_rc'])
ind_nec(df, 'GI1_3', ['ct_ic'])

clima = ['Tropical húmedo', 'Subtropical húmedo', 'Ceja de montaña']
df['GI1_4'] = pd.NA
elig = (df['rural'] == 1) & (df['clima'].isin(clima))
df.loc[elig, 'GI1_4'] = 0
df.loc[elig & df['ct_st_d'].gt(0), 'GI1_4'] = 1
df['GI1_4'] = df['GI1_4'].astype('Int8')

ind_nec(df, 'GI1_5', ['ct_cp'])

# GI2
ind_nec(df, 'GI2_2', ['ct_cad'])

# GI3
ind_nec(df, 'GI3_1', ['ct_ce'])
ind_nec(df, 'GI3_2', ['ct_st_me', 'ct_sp_me', 'ct_ri_me', 'ct_rc_me', 
                      'ct_ic_me', 'ct_amp_me'])
ind_nec(df, 'GI3_3', ['ct_ene'])

df['cu'] = df['cu_atyoe'].min()
df['fcr'] = np.where(df['urbano'] == 1, 1, 0.94)
df['area'] = df['areatech'] - (df['areadem'] + df['areari'] + df['arearc']) + \
    0.0001
df['val_act'] = df['area'] * df['cu'] * df['fcr'] * 1.05 * 1.20 * 1.18
df['GI3_4'] = df['val_act'] * 0.01
mask = ((df['finfo'] == 'Sin información') | (df['prioridad'] == '1: Riesgo'))
df.loc[mask, 'GI3_4'] = np.nan
df.drop(columns=['cu', 'fcr', 'area', 'val_act'], inplace=True)

# GI4
ind_nec(df, 'GI4_1', ['ct_sp_r'])
ind_nec(df, 'GI4_3', ['ct_acc'])
ind_nec(df, 'GI4_4', ['ct_amp'])

# GI5
ind_nec(df, 'GI5_1', ['ct_st_ra','ct_st_ae','ct_st_aad'])

##### B. Acceso adecuado a servicios básicos

In [187]:
file = 'data/LE_SSBB_2024.dta'
sb_0 = pd.read_stata(file)
sb_0 = sb_0[['cod_local', 'agua', 'des', 'ene']]
sb = sb_0.copy()

# Corrigiendo codigos locales
codlocal(sb, 'cod_local', 'codlocal')

In [188]:
sb['agua'] = sb['agua'].astype('string')
sb['des'] = sb['des'].astype('string')
sb['ene'] = sb['ene'].astype('string')

# Identificando acceso adecuado
adec = ['1. Red pública (agua potable)', 
        '2. Pilón de uso público (agua potable)',
        '3. Camión cisterna u otro similar']
sb['agua_adecuado'] = sb['agua'].isin(adec).astype('Int8')

adec = ['1. Red pública',
        '3. Tanque séptico',
        '4. Biodigestor',
        '6. Unidades básicas de saneamiento (U.B.S)']
sb['sane_adecuado'] = sb['des'].isin(adec).astype('Int8')

sb['GI2_1'] = ((sb['agua_adecuado'] == 1) & 
               (sb['sane_adecuado'] == 1)).astype('Int8')

adec = ['1. Red pública (de una empresa distribuidora de energía eléctrica)',
        '3. Paneles solares (energía fotovoltaica)',
        '4. Energía eólica']
sb['GI4_2'] = sb['ene'].isin(adec).astype('Int8')

# Eliminando observaciones sin información de servicios básicos
sb.loc[sb['agua'] == '9. Sin información', 'GI2_1'] = pd.NA
sb.loc[sb['des'] == '9. Sin información', 'GI2_1'] = pd.NA
sb.loc[sb['ene'] == '7. Sin información', 'GI4_2'] = pd.NA

# Seleccionando variables
sb_vf = sb[['codlocal', 'GI2_1', 'GI4_2']]
sb_vf.sort_values('codlocal', inplace=True)
sb_vf = sb_vf.reset_index(drop=True)

##### C. Consolidación

In [189]:
# Uniendo bases
df2 = pd.merge(df, sb_vf, on='codlocal', how='left')

# Seleccionando variables
df = df2[['codlocal', 'brecha', 'prioridad', 'finfo',
             'GI1_1', 'GI1_2', 'GI1_3', 'GI1_4', 'GI1_5',
             'GI2_1', 'GI2_2',
             'GI3_1', 'GI3_2', 'GI3_3', 'GI3_4',
             'GI4_1', 'GI4_2', 'GI4_3', 'GI4_4',
             'GI5_1']]

#### 8.1.2 Indicadores de productos

In [190]:
df = df.copy()

In [191]:
# GI1
iden_nec_prod(df, ['GI1_1', 'GI1_2', 'GI1_3', 'GI1_4', 'GI1_5'], 'GI1')

# GI2
iden_nec_prod(df, ['GI2_1', 'GI2_2'], 'GI2')
df.loc[((df['GI2_1'] == 0) & (df['GI2_2'] == 0)), 'GI2'] = 1

# GI3
iden_nec_prod(df, ['GI3_1', 'GI3_2', 'GI3_3', 'GI3_4'], 'GI3')

# GI4
iden_nec_prod(df, ['GI4_1', 'GI4_2', 'GI4_3', 'GI4_4'], 'GI4')
df.loc[((df['GI4_2'] == 0) & (df['GI4'] == 0)), 'GI4'] = 1

### 8.2 Actividades

In [192]:
# Universo de LLLEE del año
file = 'data/LE_BasePr_2024.dta'
df_0 = pd.read_stata(file)

# Lista de LLEE "existentes"
file = 'data/procesadas/llee_existentes.csv'
exis_0 = pd.read_csv(file, dtype={'codlocal': str})

In [193]:
sfl = df_0.copy()
exis = exis_0.copy()

# Corrigiendo codigos locales
codlocal(sfl, 'cod_local', 'codlocal')

# Identificando LLEE "existentes" y "nuevos"
sfl = pd.merge(sfl, exis,
               how='left',
               on='codlocal',
               indicator='union')
print(sfl['union'].value_counts(dropna=False), '\n')

# Estado del SFL de los predios
freq(sfl, 'disafil')

union
both          47955
left_only      7470
right_only        0
Name: count, dtype: int64 

                 Freq    (%)
disafil                     
No saneado     29,687  53.6%
Saneado        21,128  38.1%
No registrado   4,610   8.3%
TOTAL          55,425   100% 



#### 8.2.1 ET.1 SFL de predios existentes

In [194]:
sfl.loc[sfl['union'] == 'both', 'ET1'] = 0
sfl.loc[(sfl['ET1'] == 0) & (sfl['disafil'] == 'Saneado'), 'ET1'] = 1
sfl.loc[sfl['disafil'] == 'No registrado', 'ET1'] = np.nan

freq(sfl, 'ET1')

          Freq    (%)
ET1                  
0.0     24,841  44.8%
1.0     19,768  35.7%
 NaN    10,816  19.5%
 TOTAL  55,425   100% 



#### 8.2.2 ET.2 SFL de predios nuevos

In [195]:
sfl.loc[sfl['union'] == 'left_only', 'ET2'] = 0
sfl.loc[(sfl['ET2'] == 0) & (sfl['disafil'] == 'Saneado'), 'ET2'] = 1
sfl.loc[sfl['disafil'] == 'No registrado', 'ET2'] = np.nan

freq(sfl, 'ET2')

          Freq    (%)
ET2                  
 NaN    49,219  88.8%
0.0      4,846   8.7%
1.0      1,360   2.5%
 TOTAL  55,425   100% 



#### 8.2.3 Consolidación

In [196]:
# Seleccionando variables
sfl = sfl[['codlocal', 'ET1', 'ET2']]

# Consolidando información
df = pd.merge(df, sfl,
              on='codlocal',
              how='left')

### 8.3 Resultados específicos

In [197]:
iden_nec_prod(df, ['GI1', 'GI2', 'GI3_1', 'GI3_2', 'GI4', 'GI5_1', 'ET1'],
              'OE1')
iden_nec_prod(df, ['GI3_3', 'GI3_4'],
              'OE4')

### 8.4 Resultados finales

In [198]:
codlocal(df_0, 'cod_local', 'codlocal')
df = pd.merge(df, df_0[['codlocal', 'matricula']],
              how='left', on='codlocal')

df['RF1'] = 0
df.loc[df['finfo'] == 'Brecha Cerrada', 'RF1'] = 1

df['RF2_matri'] = 0
df.loc[df['RF1'] == 1, 'RF2_matri'] = df['matricula']

df[['matricula', 'RF2_matri']].sum()

matricula    6430260
RF2_matri     866800
dtype: int64

In [199]:
# Añadiendo año del diagnóstico
sufix = 2024
df.rename(columns={c: f'{c}_{sufix}' for c in df.columns[1:]}, 
             inplace=True)

# Exportando a csv
df.sort_values('codlocal', inplace=True)
df = df.reset_index(drop=True)
df.to_csv('data/procesadas/necint_2024.csv', index=False)
# df

## 09. Año 2025

### 9.1 Productos

#### 9.1.1 Indicadores de subproductos

##### A. Necesidad de intervenciones en infraestructura

In [200]:
file = 'data/LE_BasePr_2025.dta'
df_0 = pd.read_stata(file)
df = df_0.copy()

# Corrigiendo codigos locales
codlocal(df, 'cod_local', 'codlocal')

In [201]:
# GI1
ind_nec(df, 'GI1_1', ['ct_st_d', 'ct_sp_d'])
ind_nec(df, 'GI1_2', ['ct_ri', 'ct_rc'])
ind_nec(df, 'GI1_3', ['ct_ic'])

clima = ['Tropical húmedo', 'Subtropical húmedo', 'Ceja de montaña']
df['GI1_4'] = pd.NA
elig = (df['rural'] == 1) & (df['clima'].isin(clima))
df.loc[elig, 'GI1_4'] = 0
df.loc[elig & df['ct_st_d'].gt(0), 'GI1_4'] = 1
df['GI1_4'] = df['GI1_4'].astype('Int8')

ind_nec(df, 'GI1_5', ['ct_cp'])

# GI2
ind_nec(df, 'GI2_2', ['ct_cad'])

# GI3
ind_nec(df, 'GI3_1', ['ct_ce'])
ind_nec(df, 'GI3_2', ['ct_st_me', 'ct_sp_me', 'ct_ri_me', 'ct_rc_me', 
                      'ct_ic_me', 'ct_amp_me'])
ind_nec(df, 'GI3_3', ['ct_ene'])

df['cu'] = df['cu_atyoe'].min()
df['fcr'] = np.where(df['urbano'] == 1, 1, 0.94)
df['area'] = df['areatech'] - (df['areadem'] + df['areari'] + df['arearc']) + \
    0.0001
df['val_act'] = df['area'] * df['cu'] * df['fcr'] * 1.05 * 1.20 * 1.18
df['GI3_4'] = df['val_act'] * 0.01
mask = ((df['finfo'] == 'Sin información') | (df['prioridad'] == '1: Riesgo'))
df.loc[mask, 'GI3_4'] = np.nan
df.drop(columns=['cu', 'fcr', 'area', 'val_act'], inplace=True)

# GI4
ind_nec(df, 'GI4_1', ['ct_sp_r'])
ind_nec(df, 'GI4_3', ['ct_acc'])
ind_nec(df, 'GI4_4', ['ct_amp'])

# GI5
ind_nec(df, 'GI5_1', ['ct_st_ra','ct_st_ae','ct_st_aad'])

##### B. Acceso adecuado a servicios básicos

In [14]:
file = 'data/LE_SSBB_2025.xlsx'
sb_0 = pd.read_excel(file, sheet_name='BaseLE')
sb_0 = sb_0[['Código Local', 
             'Procedencia de abastecimiento - Agua',
             'Tipo de conexión - Alcantarillado Sanitario',
             'Procedencia de abastecimiento - Energía eléctrica']]
sb = sb_0.copy()

# Corrigiendo codigos locales
codlocal(sb, 'Código Local', 'codlocal')

In [16]:
# Identificando acceso adecuado
adec = ['1. Red pública (agua potable)', 
        '2. Pilón de uso público (agua potable)',
        '3. Camión cisterna u otro similar']
sb['agua_adecuado'] = sb['Procedencia de abastecimiento - Agua'].isin(adec). \
    astype('Int8')

adec = ['1. Red pública',
        '3. Tanque séptico',
        '4. Biodigestor',
        '6. Unidades básicas de saneamiento (U.B.S)']
sb['sane_adecuado'] = sb['Tipo de conexión - Alcantarillado Sanitario']. \
    isin(adec).astype('Int8')

sb['GI2_1'] = ((sb['agua_adecuado'] == 1) & 
                      (sb['sane_adecuado'] == 1)).astype('Int8')

adec = ['1. Red pública (de una empresa distribuidora de energía eléctrica)',
        '3. Paneles solares (energía fotovoltaica)',
        '4. Energía eólica']
sb['GI4_2'] = sb['Procedencia de abastecimiento - Energía eléctrica']. \
    isin(adec).astype('Int8')

# Eliminando observaciones sin información de servicios básicos
sb.loc[sb['Procedencia de abastecimiento - Agua'] == '9. Sin información',
       'GI2_1'] = pd.NA
sb.loc[sb['Tipo de conexión - Alcantarillado Sanitario'] == \
       '9. Sin información', 'GI2_1'] = pd.NA
sb.loc[sb['Procedencia de abastecimiento - Energía eléctrica'] == \
       '7. Sin información', 'GI4_2'] = pd.NA

# Seleccionando variables
sb_vf = sb[['codlocal', 'GI2_1', 'GI4_2']]
sb_vf.sort_values('codlocal', inplace=True)
sb_vf = sb_vf.reset_index(drop=True)

##### C. Consolidación

In [204]:
# Uniendo bases
df2 = pd.merge(df, sb_vf, on='codlocal', how='left')

# Seleccionando variables
df = df2[['codlocal', 'brecha', 'prioridad', 'finfo',
             'GI1_1', 'GI1_2', 'GI1_3', 'GI1_4', 'GI1_5',
             'GI2_1', 'GI2_2',
             'GI3_1', 'GI3_2', 'GI3_3', 'GI3_4',
             'GI4_1', 'GI4_2', 'GI4_3', 'GI4_4',
             'GI5_1']]

#### 9.1.2 Indicadores de productos

In [205]:
df = df.copy()

In [206]:
# GI1
iden_nec_prod(df, ['GI1_1', 'GI1_2', 'GI1_3', 'GI1_4', 'GI1_5'], 'GI1')

# GI2
iden_nec_prod(df, ['GI2_1', 'GI2_2'], 'GI2')
df.loc[((df['GI2_1'] == 0) & (df['GI2_2'] == 0)), 'GI2'] = 1

# GI3
iden_nec_prod(df, ['GI3_1', 'GI3_2', 'GI3_3', 'GI3_4'], 'GI3')

# GI4
iden_nec_prod(df, ['GI4_1', 'GI4_2', 'GI4_3', 'GI4_4'], 'GI4')
df.loc[((df['GI4_2'] == 0) & (df['GI4'] == 0)), 'GI4'] = 1

### 9.2 Actividades

In [207]:
# Universo de LLLEE del año
file = 'data/LE_BasePr_2025.dta'
df_0 = pd.read_stata(file)

# Lista de LLEE "existentes"
file = 'data/procesadas/llee_existentes.csv'
exis_0 = pd.read_csv(file, dtype={'codlocal': str})

In [208]:
sfl = df_0.copy()
exis = exis_0.copy()

# Corrigiendo codigos locales
codlocal(sfl, 'cod_local', 'codlocal')

# Identificando LLEE "existentes" y "nuevos"
sfl = pd.merge(sfl, exis,
               how='left',
               on='codlocal',
               indicator='union')
print(sfl['union'].value_counts(dropna=False), '\n')

# Estado del SFL de los predios
freq(sfl, 'disafil')

union
both          47951
left_only      7651
right_only        0
Name: count, dtype: int64 

                 Freq    (%)
disafil                     
No saneado     31,016  55.8%
Saneado        21,475  38.6%
No registrado   3,111   5.6%
TOTAL          55,602   100% 



#### 9.2.1 ET.1 SFL de predios existentes

In [209]:
sfl.loc[sfl['union'] == 'both', 'ET1'] = 0
sfl.loc[(sfl['ET1'] == 0) & (sfl['disafil'] == 'Saneado'), 'ET1'] = 1
sfl.loc[sfl['disafil'] == 'No registrado', 'ET1'] = np.nan

freq(sfl, 'ET1')

          Freq    (%)
ET1                  
0.0     25,788  46.4%
1.0     20,022  36.0%
 NaN     9,792  17.6%
 TOTAL  55,602   100% 



#### 9.2.2 ET.2 SFL de predios nuevos

In [210]:
sfl.loc[sfl['union'] == 'left_only', 'ET2'] = 0
sfl.loc[(sfl['ET2'] == 0) & (sfl['disafil'] == 'Saneado'), 'ET2'] = 1
sfl.loc[sfl['disafil'] == 'No registrado', 'ET2'] = np.nan

freq(sfl, 'ET2')

          Freq    (%)
ET2                  
 NaN    48,921  88.0%
0.0      5,228   9.4%
1.0      1,453   2.6%
 TOTAL  55,602   100% 



#### 9.2.3 Consolidación

In [211]:
# Seleccionando variables
sfl = sfl[['codlocal', 'ET1', 'ET2']]

# Consolidando información
df = pd.merge(df, sfl,
              on='codlocal',
              how='left')

### 9.3 Resultados específicos

In [212]:
iden_nec_prod(df, ['GI1', 'GI2', 'GI3_1', 'GI3_2', 'GI4', 'GI5_1', 'ET1'],
              'OE1')
iden_nec_prod(df, ['GI3_3', 'GI3_4'],
              'OE4')

### 9.4 Resultados finales

In [213]:
codlocal(df_0, 'cod_local', 'codlocal')
df = pd.merge(df, df_0[['codlocal', 'matricula']],
              how='left', on='codlocal')

df['RF1'] = 0
df.loc[df['finfo'] == 'Brecha Cerrada', 'RF1'] = 1

df['RF2_matri'] = 0
df.loc[df['RF1'] == 1, 'RF2_matri'] = df['matricula']

df[['matricula', 'RF2_matri']].sum()

matricula    6429493
RF2_matri     921586
dtype: int64

In [214]:
# Añadiendo año del diagnóstico
sufix = 2025
df.rename(columns={c: f'{c}_{sufix}' for c in df.columns[1:]}, 
             inplace=True)

# Exportando a csv
df.sort_values('codlocal', inplace=True)
df = df.reset_index(drop=True)
df.to_csv('data/procesadas/necint_2025.csv', index=False)

## 10. Necesidades de intervención en las líneas base

#### PNIE

In [392]:
lb = pd.read_csv('data/procesadas/lb_2017.csv', dtype={'codlocal': str})
ni = pd.read_csv('data/procesadas/necint_PNIE.csv', dtype={'codlocal': str})

df = pd.merge(lb, ni, 
              on='codlocal', how='left')
df.sort_values('codlocal', inplace=True)
df = df[['codlocal',
         'GI1_1_PNIE', 'GI1_2_PNIE', 'GI1_3_PNIE', 'GI1_4_PNIE', 'GI1_5_PNIE', 
         'GI2_1_PNIE', 'GI2_2_PNIE',
         'GI3_1_PNIE', 'GI3_2_PNIE', 'GI3_3_PNIE', 'GI3_4_PNIE', 
         'GI4_1_PNIE', 'GI4_2_PNIE', 'GI4_3_PNIE', 'GI4_4_PNIE', 
         'GI5_1_PNIE']]
df = df.reset_index(drop=True)
df.to_csv('data/procesadas/acumulado_subprod_PNIE.csv', index=False)

# Estimando la cantidad de locales educativos con necesidad de intervención
df[['GI1_1_PNIE', 'GI1_2_PNIE', 'GI1_3_PNIE', 'GI1_4_PNIE', 'GI1_5_PNIE', 
    'GI2_1_PNIE', 'GI2_2_PNIE',
    'GI3_1_PNIE', 'GI3_2_PNIE', 'GI3_3_PNIE', 'GI3_4_PNIE', 
    'GI4_1_PNIE', 'GI4_2_PNIE', 'GI4_3_PNIE', 'GI4_4_PNIE', 
    'GI5_1_PNIE']].sum()

GI1_1_PNIE        27,601.0
GI1_2_PNIE         9,808.0
GI1_3_PNIE         6,410.0
GI1_4_PNIE           944.0
GI1_5_PNIE        12,242.0
GI2_1_PNIE        12,287.0
GI2_2_PNIE        39,918.0
GI3_1_PNIE        39,411.0
GI3_2_PNIE        39,345.0
GI3_3_PNIE        34,227.0
GI3_4_PNIE   250,930,579.5
GI4_1_PNIE         8,998.0
GI4_2_PNIE        29,884.0
GI4_3_PNIE        40,685.0
GI4_4_PNIE        23,998.0
GI5_1_PNIE        17,748.0
dtype: float64

#### 2021

In [393]:
lb = pd.read_csv('data/procesadas/lb_2021.csv', dtype={'codlocal': str})
ni_PNIE = pd.read_csv('data/procesadas/necint_PNIE.csv', 
                      dtype={'codlocal': str})
ni_2019 = pd.read_csv('data/procesadas/necint_2019.csv', 
                      dtype={'codlocal': str})
ni_2020 = pd.read_csv('data/procesadas/necint_2020.csv', 
                      dtype={'codlocal': str})
ni_2021 = pd.read_csv('data/procesadas/necint_2021.csv', 
                      dtype={'codlocal': str})

In [394]:
df = pd.merge(lb, ni_PNIE,
              on='codlocal', how='left')
df = pd.merge(df, ni_2019,
              on='codlocal', how='left')
df = pd.merge(df, ni_2020,
              on='codlocal', how='left')
df = pd.merge(df, ni_2021,
              on='codlocal', how='left')
df.sort_values('codlocal', inplace=True)
df = df[['codlocal',
         'GI1_1_PNIE', 'GI1_2_PNIE', 'GI1_3_PNIE', 'GI1_4_PNIE', 'GI1_5_PNIE', 
         'GI2_1_PNIE', 'GI2_2_PNIE',
         'GI3_1_PNIE', 'GI3_2_PNIE', 'GI3_3_PNIE', 'GI3_4_PNIE', 
         'GI4_1_PNIE', 'GI4_2_PNIE', 'GI4_3_PNIE', 'GI4_4_PNIE', 
         'GI5_1_PNIE',
         'GI1_1_2019', 'GI1_2_2019', 'GI1_3_2019', 'GI1_4_2019', 'GI1_5_2019', 
         'GI2_1_2019', 'GI2_2_2019',
         'GI3_1_2019', 'GI3_2_2019', 'GI3_3_2019', 'GI3_4_2019', 
         'GI4_1_2019', 'GI4_2_2019', 'GI4_3_2019', 'GI4_4_2019', 
         'GI5_1_2019', 
         'GI1_1_2020', 'GI1_2_2020', 'GI1_3_2020', 'GI1_4_2020', 'GI1_5_2020', 
         'GI2_1_2020', 'GI2_2_2020',
         'GI3_1_2020', 'GI3_2_2020', 'GI3_3_2020', 'GI3_4_2020', 
         'GI4_1_2020', 'GI4_2_2020', 'GI4_3_2020', 'GI4_4_2020', 
         'GI5_1_2020', 
         'GI1_1_2021', 'GI1_2_2021', 'GI1_3_2021', 'GI1_4_2021', 'GI1_5_2021', 
         'GI2_1_2021', 'GI2_2_2021',
         'GI3_1_2021', 'GI3_2_2021', 'GI3_3_2021', 'GI3_4_2021', 
         'GI4_1_2021', 'GI4_2_2021', 'GI4_3_2021', 'GI4_4_2021', 
         'GI5_1_2021']]

In [395]:
def iden_nec(data, ind, anios):
    vars = [ind + '_' + i for i in anios]
    data[ind] = data[vars].eq(1).any(axis=1).astype('int8')

In [396]:
anios = ['PNIE', '2019', '2020', '2021']
inds = ['GI1_1', 'GI1_2', 'GI1_3', 'GI1_4', 'GI1_5', 
        'GI2_1', 'GI2_2', 
        'GI3_1', 'GI3_2', 'GI3_3', 'GI3_4', 
        'GI4_1', 'GI4_2', 'GI4_3', 'GI4_4', 
        'GI5_1']
for ind in inds:
    iden_nec(df, ind, anios)

In [397]:
df = df.reset_index(drop=True)
df.to_csv('data/procesadas/acumulado_subprod_2021.csv', index=False)

# Estimando la cantidad de locales educativos con necesidad de intervención
df[['GI1_1', 'GI1_2', 'GI1_3', 'GI1_4', 'GI1_5', 
    'GI2_1', 'GI2_2', 
    'GI3_1', 'GI3_2', 'GI3_3', 'GI3_4', 
    'GI4_1', 'GI4_2', 'GI4_3', 'GI4_4', 
    'GI5_1']].sum()

GI1_1    33326
GI1_2    11431
GI1_3     7632
GI1_4     9697
GI1_5    15470
GI2_1    33781
GI2_2    42411
GI3_1    41853
GI3_2    45971
GI3_3    35965
GI3_4        0
GI4_1     9821
GI4_2    50164
GI4_3    42874
GI4_4    26768
GI5_1    24260
dtype: int64

#### 2025

In [398]:
lb = pd.read_csv('data/procesadas/lb_2025.csv', dtype={'codlocal': str})
ni_PNIE = pd.read_csv('data/procesadas/necint_PNIE.csv', 
                      dtype={'codlocal': str})
ni_2019 = pd.read_csv('data/procesadas/necint_2019.csv', 
                      dtype={'codlocal': str})
ni_2020 = pd.read_csv('data/procesadas/necint_2020.csv', 
                      dtype={'codlocal': str})
ni_2021 = pd.read_csv('data/procesadas/necint_2021.csv', 
                      dtype={'codlocal': str})
ni_2022 = pd.read_csv('data/procesadas/necint_2022.csv', 
                      dtype={'codlocal': str})
ni_2023 = pd.read_csv('data/procesadas/necint_2023.csv', 
                      dtype={'codlocal': str})
ni_2024 = pd.read_csv('data/procesadas/necint_2024.csv', 
                      dtype={'codlocal': str})
ni_2025 = pd.read_csv('data/procesadas/necint_2025.csv', 
                      dtype={'codlocal': str})

In [399]:
df = pd.merge(lb, ni_PNIE,
              on='codlocal', how='left')
df = pd.merge(df, ni_2019,
              on='codlocal', how='left')
df = pd.merge(df, ni_2020,
              on='codlocal', how='left')
df = pd.merge(df, ni_2021,
              on='codlocal', how='left')
df = pd.merge(df, ni_2022,
              on='codlocal', how='left')
df = pd.merge(df, ni_2023,
              on='codlocal', how='left')
df = pd.merge(df, ni_2024,
              on='codlocal', how='left')
df = pd.merge(df, ni_2025,
              on='codlocal', how='left')
df.sort_values('codlocal', inplace=True)
df = df[['codlocal',
         'GI1_1_PNIE', 'GI1_2_PNIE', 'GI1_3_PNIE', 'GI1_4_PNIE', 'GI1_5_PNIE', 
         'GI2_1_PNIE', 'GI2_2_PNIE',
         'GI3_1_PNIE', 'GI3_2_PNIE', 'GI3_3_PNIE', 'GI3_4_PNIE', 
         'GI4_1_PNIE', 'GI4_2_PNIE', 'GI4_3_PNIE', 'GI4_4_PNIE', 
         'GI5_1_PNIE',
         'GI1_1_2019', 'GI1_2_2019', 'GI1_3_2019', 'GI1_4_2019', 'GI1_5_2019', 
         'GI2_1_2019', 'GI2_2_2019',
         'GI3_1_2019', 'GI3_2_2019', 'GI3_3_2019', 'GI3_4_2019', 
         'GI4_1_2019', 'GI4_2_2019', 'GI4_3_2019', 'GI4_4_2019', 
         'GI5_1_2019', 
         'GI1_1_2020', 'GI1_2_2020', 'GI1_3_2020', 'GI1_4_2020', 'GI1_5_2020', 
         'GI2_1_2020', 'GI2_2_2020',
         'GI3_1_2020', 'GI3_2_2020', 'GI3_3_2020', 'GI3_4_2020', 
         'GI4_1_2020', 'GI4_2_2020', 'GI4_3_2020', 'GI4_4_2020', 
         'GI5_1_2020', 
         'GI1_1_2021', 'GI1_2_2021', 'GI1_3_2021', 'GI1_4_2021', 'GI1_5_2021', 
         'GI2_1_2021', 'GI2_2_2021',
         'GI3_1_2021', 'GI3_2_2021', 'GI3_3_2021', 'GI3_4_2021', 
         'GI4_1_2021', 'GI4_2_2021', 'GI4_3_2021', 'GI4_4_2021', 
         'GI5_1_2021',
         'GI1_1_2022', 'GI1_2_2022', 'GI1_3_2022', 'GI1_4_2022', 'GI1_5_2022',
         'GI2_1_2022', 'GI2_2_2022',
         'GI3_1_2022', 'GI3_2_2022', 'GI3_3_2022', 'GI3_4_2022',
         'GI4_1_2022', 'GI4_2_2022', 'GI4_3_2022', 'GI4_4_2022',
         'GI5_1_2022',
         'GI1_1_2023', 'GI1_2_2023', 'GI1_3_2023', 'GI1_4_2023', 'GI1_5_2023',
         'GI2_1_2023', 'GI2_2_2023',
         'GI3_1_2023', 'GI3_2_2023', 'GI3_3_2023', 'GI3_4_2023',
         'GI4_1_2023', 'GI4_2_2023', 'GI4_3_2023', 'GI4_4_2023',
         'GI5_1_2023',
         'GI1_1_2024', 'GI1_2_2024', 'GI1_3_2024', 'GI1_4_2024', 'GI1_5_2024',
         'GI2_1_2024', 'GI2_2_2024',
         'GI3_1_2024', 'GI3_2_2024', 'GI3_3_2024', 'GI3_4_2024',
         'GI4_1_2024', 'GI4_2_2024', 'GI4_3_2024', 'GI4_4_2024', 
         'GI5_1_2024',
         'GI1_1_2025', 'GI1_2_2025', 'GI1_3_2025', 'GI1_4_2025', 'GI1_5_2025',
         'GI2_1_2025', 'GI2_2_2025',
         'GI3_1_2025', 'GI3_2_2025', 'GI3_3_2025', 'GI3_4_2025',
         'GI4_1_2025', 'GI4_2_2025', 'GI4_3_2025', 'GI4_4_2025',
         'GI5_1_2025']]

In [400]:
anios = ['PNIE', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
inds = ['GI1_1', 'GI1_2', 'GI1_3', 'GI1_4', 'GI1_5', 
        'GI2_1', 'GI2_2', 
        'GI3_1', 'GI3_2', 'GI3_3', 'GI3_4', 
        'GI4_1', 'GI4_2', 'GI4_3', 'GI4_4', 
        'GI5_1']
for ind in inds:
    iden_nec(df, ind, anios)

In [401]:
df = df.reset_index(drop=True)
df.to_csv('data/procesadas/acumulado_subprod_2025.csv', index=False)

# Estimando la cantidad de locales educativos con necesidad de intervención
df[['GI1_1', 'GI1_2', 'GI1_3', 'GI1_4', 'GI1_5', 
    'GI2_1', 'GI2_2', 
    'GI3_1', 'GI3_2', 'GI3_3', 'GI3_4', 
    'GI4_1', 'GI4_2', 'GI4_3', 'GI4_4', 
    'GI5_1']].sum()

GI1_1    46656
GI1_2    15742
GI1_3    11592
GI1_4    16732
GI1_5    16757
GI2_1    38005
GI2_2    47261
GI3_1    46816
GI3_2    54794
GI3_3    42591
GI3_4        0
GI4_1    13305
GI4_2    52136
GI4_3    47325
GI4_4    33627
GI5_1    40382
dtype: int64